# 1\.2\.3 Standardize and Aggregate TLC Datasets

The TLC datasets are a different beast than the traffic and bus datasets\. While those datasets arrive in a partially aggregated form, TLC records every individual trip\. Across the retained Yellow Taxi, Green Taxi, and FHVHV datasets, that adds up to nearly one billion trips\. An earlier attempt to simply add temporal fields and carry the trip\-level records forward quickly exposed the downside of that approach: large storage requirements, long processing times, and little practical benefit for the analyses we actually plan to perform\.

Rather than creating multiple generations of massive intermediate files, this notebook standardizes and aggregates the TLC data in a single pass\. Each trip is assigned its canonical pickup\-based timestamp, enriched with the project's shared temporal dimensions, and aggregated into Taxi Zone\-level mobility summaries\. Because the downstream anomaly detection, clustering, and mobility regime analyses all operate on aggregated mobility states rather than individual trips, this approach dramatically reduces data volume while preserving the information we actually need\.

One design choice worth calling out is that we retain the hour\-level representation rather than aggregating directly into temporal buckets\. Temporal buckets can always be constructed later, but once hourly detail is lost it cannot be recovered\. Keeping the hour gives us maximum flexibility for future aggregation strategies while still supporting the project's standard temporal bucket framework\.


In [1]:
from pathlib import Path
import gc
import json
import shutil
import time

import duckdb
import pandas as pd
import pyarrow.parquet as pq


In [2]:
SOURCE_DATA_DIR = Path("source_data")
PIPELINE_DATA_DIR = Path("pipeline_data")

TLC_INTERMEDIATE_OUTPUT_DIR = PIPELINE_DATA_DIR / "1.2.3.intermediate_tables"
TLC_FINAL_OUTPUT_DIR = PIPELINE_DATA_DIR / "1.2.3.final_tables"
TLC_FINAL_CONTRACT_DIR = PIPELINE_DATA_DIR / "1.2.3.final_tlc_contract"

TLC_OD_FINAL_OUTPUT_DIR = TLC_FINAL_CONTRACT_DIR / "tlc_od_mobility_parquet"
TLC_PICKUP_TEMPORAL_FINAL_OUTPUT_DIR = TLC_FINAL_CONTRACT_DIR / "tlc_pickup_temporal_mobility_parquet"
TLC_OD_FINAL_OUTPUT_GLOB = TLC_OD_FINAL_OUTPUT_DIR / "*.parquet"
TLC_PICKUP_TEMPORAL_FINAL_OUTPUT_GLOB = TLC_PICKUP_TEMPORAL_FINAL_OUTPUT_DIR / "*.parquet"
TLC_FINAL_MODALITY_CONTRACT_PATH = TLC_FINAL_CONTRACT_DIR / "tlc_final_modality_contract.json"
TLC_FINAL_CONTRACT_VALIDATION_PATH = TLC_INTERMEDIATE_OUTPUT_DIR / "tlc_final_contract_validation.csv"

SOURCE_DATA_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_DATA_DIR.mkdir(parents=True, exist_ok=True)
TLC_INTERMEDIATE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TLC_FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TLC_FINAL_CONTRACT_DIR.mkdir(parents=True, exist_ok=True)
TLC_OD_FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TLC_PICKUP_TEMPORAL_FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Master Control Panel

RUN_RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC = False
RUN_TLC_MOBILITY_TABLE_GENERATION = False
FILTER_INVALID_TLC_TRIPS = True
RUN_TLC_MOBILITY_METRIC_QUALITY_VALIDATION = False

# One-time repair toggle for existing monthly TLC mobility outputs.
# Leave this False during normal runs. Set True only when intentionally rebuilding
# every monthly mobility file with the service-month date boundary enforced upstream.
REBUILD_ALL_TLC_MOBILITY_TABLES_WITH_SERVICE_MONTH_FILTER = False

# Optional targeted repair controls for smaller reruns.
# Examples: ["2023-07", "2026-04"] or [("yellow", "2023-07")]
REBUILD_TLC_MOBILITY_SERVICE_MONTHS = []
REBUILD_TLC_MOBILITY_DATASET_MONTHS = []

# Rebuild the final Taxi/FHVHV TLC contract even if the outputs already exist.
REBUILD_FINAL_TLC_MODALITY_CONTRACT = False
RUN_FINAL_TLC_MODALITY_CONTRACT_VALIDATION = False

# Optional audit-only output. The required final handoff is the compact pickup-temporal table.
WRITE_FINAL_TLC_OD_AUDIT_OUTPUT = False

MAX_VALID_TLC_TRIP_DURATION_SECONDS = 86_400
MIN_VALID_TLC_TRIP_DURATION_SECONDS = 60
MIN_VALID_TLC_TRIP_DISTANCE = 0
MAX_VALID_TLC_TRIP_DISTANCE = 100
MAX_VALID_TLC_TRIP_SPEED_MPH = 100

## 1\.2\.3\.1  Generate TLC Spatiotemporal Mobility Tables

This section turns raw TLC trip records into compact spatiotemporal mobility tables\. We add the temporal and spatial fields we need, then aggregate immediately so we do not carry nearly one billion trip\-level records through the rest of the pipeline\.

In [4]:
# ---------------------------------------------------------------------
# Configure TLC spatiotemporal mobility table generation
# ---------------------------------------------------------------------

TLC_MOBILITY_OUTPUT_DIR = TLC_FINAL_OUTPUT_DIR

TLC_TEMPORAL_VALIDATION_MANIFEST_PATH = (
    TLC_INTERMEDIATE_OUTPUT_DIR / "tlc_temporal_field_validation_manifest.csv"
)

DUCKDB_TEMP_DIR = TLC_INTERMEDIATE_OUTPUT_DIR / "duckdb_temp"
DUCKDB_TEMP_DIR.mkdir(parents=True, exist_ok=True)

TLC_DATA_DIR = SOURCE_DATA_DIR / "tlc"

TLC_MOBILITY_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DUCKDB_TEMP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

retained_tlc_datasets = {
    "yellow": {
        "pickup_datetime_col": "tpep_pickup_datetime",
        "dropoff_datetime_col": "tpep_dropoff_datetime",
        "pickup_zone_col": "PULocationID",
        "dropoff_zone_col": "DOLocationID",
        "distance_col": "trip_distance",
        "duration_seconds_expr": """
            DATE_DIFF(
                'second',
                tpep_pickup_datetime,
                tpep_dropoff_datetime
            )
        """,
    },
    "green": {
        "pickup_datetime_col": "lpep_pickup_datetime",
        "dropoff_datetime_col": "lpep_dropoff_datetime",
        "pickup_zone_col": "PULocationID",
        "dropoff_zone_col": "DOLocationID",
        "distance_col": "trip_distance",
        "duration_seconds_expr": """
            DATE_DIFF(
                'second',
                lpep_pickup_datetime,
                lpep_dropoff_datetime
            )
        """,
    },
    "fhvhv": {
        "pickup_datetime_col": "pickup_datetime",
        "dropoff_datetime_col": "dropoff_datetime",
        "pickup_zone_col": "PULocationID",
        "dropoff_zone_col": "DOLocationID",
        "distance_col": "trip_miles",
        "duration_seconds_expr": "trip_time",
    },
}

retained_tlc_datasets

{'yellow': {'pickup_datetime_col': 'tpep_pickup_datetime',
  'dropoff_datetime_col': 'tpep_dropoff_datetime',
  'pickup_zone_col': 'PULocationID',
  'dropoff_zone_col': 'DOLocationID',
  'distance_col': 'trip_distance',
  'duration_seconds_expr': "\n            DATE_DIFF(\n                'second',\n                tpep_pickup_datetime,\n                tpep_dropoff_datetime\n            )\n        "},
 'green': {'pickup_datetime_col': 'lpep_pickup_datetime',
  'dropoff_datetime_col': 'lpep_dropoff_datetime',
  'pickup_zone_col': 'PULocationID',
  'dropoff_zone_col': 'DOLocationID',
  'distance_col': 'trip_distance',
  'duration_seconds_expr': "\n            DATE_DIFF(\n                'second',\n                lpep_pickup_datetime,\n                lpep_dropoff_datetime\n            )\n        "},
 'fhvhv': {'pickup_datetime_col': 'pickup_datetime',
  'dropoff_datetime_col': 'dropoff_datetime',
  'pickup_zone_col': 'PULocationID',
  'dropoff_zone_col': 'DOLocationID',
  'distance_col

Before rebuilding the TLC mobility tables, we first inspect the raw trip\-level distance, duration, and implied speed distributions\. This diagnostic helps quantify how many records would be excluded by the updated quality filters and confirms that extreme derived speeds are caused by implausible source distance/duration combinations rather than downstream aggregation logic\.

In [5]:
# ---------------------------------------------------------------------
# Diagnose raw TLC distance, duration, and implied speed quality
# ---------------------------------------------------------------------

RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC_PATH = (
    TLC_INTERMEDIATE_OUTPUT_DIR / "raw_tlc_distance_duration_diagnostic.csv"
)

if RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC_PATH.exists() and not RUN_RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC:

    raw_tlc_distance_duration_diagnostic_df = pd.read_csv(
        RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC_PATH
    )

    print(
        "Loaded existing raw TLC distance/duration diagnostic:",
        RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC_PATH
    )

else:

    raw_quality_results = []

    # This diagnostic scans monthly raw TLC files and writes progress after each file.
    # That matters because the raw TLC universe is huge and a full scan can be interrupted.
    if RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC_PATH.exists():

        existing_raw_quality_df = pd.read_csv(
            RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC_PATH
        )

        processed_keys = set(
            zip(
                existing_raw_quality_df["dataset_type"],
                existing_raw_quality_df["service_month"],
            )
        )

        raw_quality_results.append(existing_raw_quality_df)

        print(
            "Resuming existing raw TLC distance/duration diagnostic:",
            RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC_PATH
        )

    else:

        processed_keys = set()

        print(
            "Starting new raw TLC distance/duration diagnostic:",
            RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC_PATH
        )

    for dataset_type, config in retained_tlc_datasets.items():

        distance_col = config["distance_col"]
        duration_seconds_expr = config["duration_seconds_expr"]

        source_files = sorted(
            TLC_DATA_DIR.glob(f"{dataset_type}_tripdata_*.parquet")
        )

        print(
            f"\nDiagnosing {dataset_type}: {len(source_files):,} monthly parquet files"
        )

        for file_index, source_path in enumerate(source_files, start=1):

            service_month = (
                source_path.name
                .split("_tripdata_")[1]
                .replace(".parquet", "")
            )

            diagnostic_key = (dataset_type, service_month)

            if diagnostic_key in processed_keys:

                print(
                    f"Skipping {dataset_type} {service_month} "
                    f"({file_index:,}/{len(source_files):,}) — already diagnosed"
                )

                continue

            print(
                f"Diagnosing {dataset_type} {service_month} "
                f"({file_index:,}/{len(source_files):,})"
            )

            # These checks evaluate the raw trip universe before aggregation.
            # The goal is to quantify impossible trips, not to remove anything in this diagnostic step.
            query = f"""
                SELECT
                    '{dataset_type}' AS dataset_type,
                    '{service_month}' AS service_month,

                    COUNT(*) AS raw_trip_rows,

                    SUM(CASE WHEN "{distance_col}" < 0 THEN 1 ELSE 0 END)
                        AS negative_distance_rows,

                    SUM(CASE WHEN "{distance_col}" > {MAX_VALID_TLC_TRIP_DISTANCE} THEN 1 ELSE 0 END)
                        AS excessive_distance_rows,

                    SUM(CASE WHEN {duration_seconds_expr} < {MIN_VALID_TLC_TRIP_DURATION_SECONDS} THEN 1 ELSE 0 END)
                        AS short_duration_rows,

                    SUM(CASE WHEN {duration_seconds_expr} > {MAX_VALID_TLC_TRIP_DURATION_SECONDS} THEN 1 ELSE 0 END)
                        AS excessive_duration_rows,

                    SUM(
                        CASE
                            WHEN {duration_seconds_expr} > 0
                             AND ("{distance_col}" / ({duration_seconds_expr} / 3600.0)) > {MAX_VALID_TLC_TRIP_SPEED_MPH}
                            THEN 1 ELSE 0
                        END
                    ) AS excessive_speed_rows,

                    MIN("{distance_col}") AS min_distance,
                    QUANTILE_CONT("{distance_col}", 0.50) AS p50_distance,
                    QUANTILE_CONT("{distance_col}", 0.95) AS p95_distance,
                    QUANTILE_CONT("{distance_col}", 0.99) AS p99_distance,
                    QUANTILE_CONT("{distance_col}", 0.999) AS p999_distance,
                    MAX("{distance_col}") AS max_distance,

                    MIN({duration_seconds_expr}) AS min_duration_seconds,
                    QUANTILE_CONT({duration_seconds_expr}, 0.50) AS p50_duration_seconds,
                    QUANTILE_CONT({duration_seconds_expr}, 0.95) AS p95_duration_seconds,
                    QUANTILE_CONT({duration_seconds_expr}, 0.99) AS p99_duration_seconds,
                    MAX({duration_seconds_expr}) AS max_duration_seconds

                FROM read_parquet('{source_path.as_posix()}')
            """

            monthly_quality_df = duckdb.sql(query).df()

            raw_quality_results.append(monthly_quality_df)

            raw_tlc_distance_duration_diagnostic_df = (
                pd.concat(raw_quality_results, ignore_index=True)
                .drop_duplicates(
                    subset=["dataset_type", "service_month"],
                    keep="last"
                )
                .sort_values(["dataset_type", "service_month"])
                .reset_index(drop=True)
            )

            raw_tlc_distance_duration_diagnostic_df.to_csv(
                RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC_PATH,
                index=False
            )

            processed_keys.add(diagnostic_key)

            print(
                f"Saved diagnostic progress through {dataset_type} {service_month}"
            )

    raw_tlc_distance_duration_diagnostic_df = pd.read_csv(
        RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC_PATH
    )

display(raw_tlc_distance_duration_diagnostic_df)

Loaded existing raw TLC distance/duration diagnostic: pipeline_data/1.2.3.intermediate_tables/raw_tlc_distance_duration_diagnostic.csv


,dataset_type,service_month,raw_trip_rows,negative_distance_rows,excessive_distance_rows,short_duration_rows,excessive_duration_rows,excessive_speed_rows,min_distance,p50_distance,p95_distance,p99_distance,p999_distance,max_distance,min_duration_seconds,p50_duration_seconds,p95_duration_seconds,p99_duration_seconds,max_duration_seconds
0,fhvhv,2023-01,18479031,0.0,1699.0,2238.0,0.0,6.0,0.0,2.897,16.060,25.79,51.150000,407.563,0,906.0,2526.0,3685.00,35359
1,fhvhv,2023-02,17960971,0.0,1266.0,1638.0,0.0,6.0,0.0,2.860,15.791,25.81,50.120000,423.000,0,914.0,2565.0,3740.00,43234
2,fhvhv,2023-03,20413539,0.0,1676.0,2162.0,0.0,8.0,0.0,2.913,16.320,26.69,52.200000,307.190,0,940.0,2744.0,4053.00,51226
3,fhvhv,2023-04,19144903,0.0,1571.0,1865.0,0.0,12.0,0.0,2.990,16.464,27.08,52.330000,362.190,0,946.0,2752.0,4094.00,63519
4,fhvhv,2023-05,19847676,0.0,1826.0,1790.0,1.0,6.0,0.0,3.066,16.860,27.49,53.444625,320.970,0,990.0,2985.0,4472.00,94853
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,yellow,2025-12,4305006,0.0,197.0,105134.0,44.0,974.0,0.0,1.810,12.700,19.55,30.400000,322576.170,-2684,876.0,2877.0,4795.95,516712
114,yellow,2026-01,3724889,0.0,162.0,83747.0,35.0,732.0,0.0,1.810,12.450,19.54,29.790000,269097.480,-702,802.0,2565.0,4286.00,450474
115,yellow,2026-02,3399866,0.0,172.0,73605.0,37.0,620.0,0.0,1.800,12.050,19.41,29.920000,328522.200,-63,841.0,2627.0,4351.00,555909
116,yellow,2026-03,3952451,0.0,161.0,89190.0,28.0,930.0,0.0,1.820,12.700,19.58,29.900000,288381.680,-11200,797.0,2680.0,4515.00,457236


Findings\. The diagnostic confirms that the TLC source data contains a small but persistent tail of invalid trip records, especially implausible distances and implied speeds\. The core distributions look reasonable — for example, yellow taxi p99 distances are generally around 19–21 miles and FHVHV p99 distances are around 25–28 miles — but the maximum distances are clearly impossible, often reaching tens or hundreds of thousands of miles\. Short\-duration records are also common, especially in yellow and green taxi files\. This supports the new filtering rules: keep the valid majority of TLC trips, but remove trips with excessive distance, very short duration, excessive duration, or impossible implied speed before aggregation\.

Before applying the new TLC quality filters, we summarize the results of the raw distance, duration, and implied\-speed diagnostic\. Records were flagged if they contained trips shorter than 60 seconds, trips longer than 24 hours, trip distances greater than 100 miles, or implied speeds greater than 100 mph\. This diagnostic helps quantify the scale of data quality issues and validate that the proposed filtering rules target a relatively small number of clearly implausible records\.

In [6]:
# ---------------------------------------------------------------------
# Summarize raw TLC distance, duration, and speed diagnostics
# ---------------------------------------------------------------------

raw_tlc_quality_summary_df = (
    raw_tlc_distance_duration_diagnostic_df
    .groupby("dataset_type", as_index=False)
    .agg(
        raw_trip_rows=("raw_trip_rows", "sum"),
        negative_distance_rows=("negative_distance_rows", "sum"),
        excessive_distance_rows=("excessive_distance_rows", "sum"),
        short_duration_rows=("short_duration_rows", "sum"),
        excessive_duration_rows=("excessive_duration_rows", "sum"),
        excessive_speed_rows=("excessive_speed_rows", "sum"),
        max_distance=("max_distance", "max"),
        min_duration_seconds=("min_duration_seconds", "min"),
        max_duration_seconds=("max_duration_seconds", "max"),
    )
)

raw_tlc_quality_summary_df["total_flagged_rows"] = (
    raw_tlc_quality_summary_df[
        [
            "negative_distance_rows",
            "excessive_distance_rows",
            "short_duration_rows",
            "excessive_duration_rows",
            "excessive_speed_rows",
        ]
    ]
    .sum(axis=1)
)

raw_tlc_quality_summary_df["pct_flagged_rows"] = (
    raw_tlc_quality_summary_df["total_flagged_rows"]
    / raw_tlc_quality_summary_df["raw_trip_rows"]
).round(4)

display(raw_tlc_quality_summary_df)

,dataset_type,raw_trip_rows,negative_distance_rows,excessive_distance_rows,short_duration_rows,excessive_duration_rows,excessive_speed_rows,max_distance,min_duration_seconds,max_duration_seconds,total_flagged_rows,pct_flagged_rows
0,fhvhv,778424569,0.0,71518.0,64259.0,1.0,237.0,5380.78,0,94853,136015.0,0.0002
1,green,2160506,0.0,1047.0,67750.0,2.0,8060.0,278990.28,-3420,90046,76859.0,0.0356
2,yellow,143080418,0.0,6418.0,2337843.0,915.0,45748.0,398608.62,-1694897908,892846,2390924.0,0.0167


Findings\. The diagnostic shows that data quality issues are concentrated in the Taxi datasets, particularly Yellow Taxi\. FHVHV data is remarkably clean, with only about 0\.02% of records flagged despite nearly 778 million trips\. Yellow Taxi contains approximately 2\.4 million flagged records \(1\.67%\), driven primarily by extremely short trip durations and implausible implied speeds\. Green Taxi shows the highest flagged rate at 3\.56%, although the dataset is much smaller\. Across all three datasets, the problematic records represent a small minority of observations, suggesting that targeted filtering should substantially improve metric quality while preserving the vast majority of trip activity\.

Now we process one monthly file at a time\. For each file, DuckDB derives the shared temporal fields, standardizes pickup and dropoff Taxi Zone identifiers, and aggregates individual trips into origin\-destination mobility states\. Each resulting record represents the number of trips and average trip characteristics observed for a specific Pickup Taxi Zone × Dropoff Taxi Zone × Date × Hour combination\. Preserving both pickup and dropoff zones allows us to retain mobility flow patterns while dramatically reducing the volume of data carried into downstream analysis\. If the notebook gets interrupted, we can rerun this cell and it will skip outputs that already exist\.

In [7]:
# ---------------------------------------------------------------------
# Generate TLC spatiotemporal mobility tables
# ---------------------------------------------------------------------

tlc_mobility_jobs = []
targeted_service_months = set(REBUILD_TLC_MOBILITY_SERVICE_MONTHS)
targeted_dataset_months = set(REBUILD_TLC_MOBILITY_DATASET_MONTHS)

for dataset_type, config in retained_tlc_datasets.items():

    source_files = sorted(
        TLC_DATA_DIR.glob(f"{dataset_type}_tripdata_*.parquet")
    )

    for source_path in source_files:

        service_month = (
            source_path.name
            .split("_tripdata_")[1]
            .replace(".parquet", "")
        )

        service_month_start = pd.Timestamp(f"{service_month}-01")
        next_service_month_start = service_month_start + pd.offsets.MonthBegin(1)

        output_path = (
            TLC_MOBILITY_OUTPUT_DIR /
            f"{dataset_type}_tripdata_{service_month}_mobility.parquet"
        )

        output_exists = (
            output_path.exists()
            and output_path.stat().st_size > 0
        )

        targeted_rebuild = (
            REBUILD_ALL_TLC_MOBILITY_TABLES_WITH_SERVICE_MONTH_FILTER
            or service_month in targeted_service_months
            or (dataset_type, service_month) in targeted_dataset_months
        )

        tlc_mobility_jobs.append({
            "dataset_type": dataset_type,
            "service_month": service_month,
            "service_month_start": service_month_start,
            "next_service_month_start": next_service_month_start,
            "source_path": source_path,
            "output_path": output_path,
            "output_exists": output_exists,
            "targeted_rebuild": targeted_rebuild,
            **config,
        })

jobs_to_process = [
    job for job in tlc_mobility_jobs
    if RUN_TLC_MOBILITY_TABLE_GENERATION
    or job["targeted_rebuild"]
    or not job["output_exists"]
]

missing_tlc_mobility_jobs = [
    job for job in tlc_mobility_jobs
    if not job["output_exists"]
]

targeted_rebuild_jobs = [
    job for job in tlc_mobility_jobs
    if job["targeted_rebuild"]
]

print("Expected TLC mobility outputs:", len(tlc_mobility_jobs))
print("Missing TLC mobility outputs:", len(missing_tlc_mobility_jobs))
print("Targeted TLC mobility rebuild outputs:", len(targeted_rebuild_jobs))
print("TLC mobility outputs to process this run:", len(jobs_to_process))

if len(jobs_to_process) > 0:

    total_jobs = len(tlc_mobility_jobs)
    processed_jobs = 0

    for i, job in enumerate(tlc_mobility_jobs, start=1):

        dataset_type = job["dataset_type"]
        service_month = job["service_month"]
        service_month_start = job["service_month_start"]
        next_service_month_start = job["next_service_month_start"]
        source_path = job["source_path"]
        output_path = job["output_path"]
        targeted_rebuild = job["targeted_rebuild"]

        pickup_datetime_col = job["pickup_datetime_col"]
        pickup_zone_col = job["pickup_zone_col"]
        dropoff_zone_col = job["dropoff_zone_col"]
        distance_col = job["distance_col"]
        duration_seconds_expr = job["duration_seconds_expr"]

        should_process_job = (
            RUN_TLC_MOBILITY_TABLE_GENERATION
            or targeted_rebuild
            or not (
                output_path.exists()
                and output_path.stat().st_size > 0
            )
        )

        if not should_process_job:
            print(
                f"[{i}/{total_jobs}] Already exists, skipping: "
                f"{output_path.name}",
                flush=True
            )
            continue

        if output_path.exists():
            output_path.unlink()

        processed_jobs += 1

        print(
            f"[{i}/{total_jobs}; processing {processed_jobs}/{len(jobs_to_process)}] "
            f"Processing {dataset_type} {service_month}: {source_path.name}",
            flush=True
        )

        # Keep each raw monthly file bounded to its service month. TLC source files can
        # contain adjacent-month pickup timestamps, and those rows are handled by their
        # own service-month files rather than duplicated or shifted downstream.
        service_month_filter = f"""
            CAST("{pickup_datetime_col}" AS DATE) >= DATE '{service_month_start.date()}'
            AND CAST("{pickup_datetime_col}" AS DATE) < DATE '{next_service_month_start.date()}'
        """

        # Filtering happens before aggregation so bad trip-level records cannot contaminate
        # average distance, duration, or implied-speed metrics in the mobility states.
        trip_quality_filter = f"""
            {duration_seconds_expr} BETWEEN {MIN_VALID_TLC_TRIP_DURATION_SECONDS}
                AND {MAX_VALID_TLC_TRIP_DURATION_SECONDS}
            AND "{distance_col}" BETWEEN {MIN_VALID_TLC_TRIP_DISTANCE}
                AND {MAX_VALID_TLC_TRIP_DISTANCE}
            AND (
                "{distance_col}" / ({duration_seconds_expr} / 3600.0)
            ) <= {MAX_VALID_TLC_TRIP_SPEED_MPH}
        """

        where_filters = [service_month_filter]

        if FILTER_INVALID_TLC_TRIPS:
            where_filters.append(trip_quality_filter)

        where_clause = "WHERE " + "\n            AND ".join(
            f"({where_filter})" for where_filter in where_filters
        )

        # This query standardizes each trip to pickup-based time and origin-destination grain,
        # then immediately aggregates to avoid carrying trip-level TLC records downstream.
        query = f"""
            COPY (
                SELECT
                    '{dataset_type}' AS dataset_type,

                    CAST("{pickup_zone_col}" AS INTEGER) AS pickup_zone_id,
                    CAST("{dropoff_zone_col}" AS INTEGER) AS dropoff_zone_id,

                    CAST("{pickup_datetime_col}" AS DATE) AS date,
                    EXTRACT(year FROM "{pickup_datetime_col}") AS year,
                    EXTRACT(month FROM "{pickup_datetime_col}") AS month,
                    CAST(
                        STRFTIME("{pickup_datetime_col}", '%w')
                        AS INTEGER
                    ) AS day_of_week,
                    EXTRACT(hour FROM "{pickup_datetime_col}") AS hour,

                    CASE
                        WHEN STRFTIME("{pickup_datetime_col}", '%w') IN ('0', '6')
                            THEN 'weekend_'
                        ELSE 'weekday_'
                    END ||
                    CASE
                        WHEN EXTRACT(hour FROM "{pickup_datetime_col}") BETWEEN 6 AND 9
                            THEN 'am_peak'
                        WHEN EXTRACT(hour FROM "{pickup_datetime_col}") BETWEEN 10 AND 15
                            THEN 'midday'
                        WHEN EXTRACT(hour FROM "{pickup_datetime_col}") BETWEEN 16 AND 19
                            THEN 'pm_peak'
                        WHEN EXTRACT(hour FROM "{pickup_datetime_col}") BETWEEN 20 AND 23
                            THEN 'evening'
                        ELSE 'overnight'
                    END AS temporal_bucket,

                    CASE
                        WHEN CAST("{pickup_datetime_col}" AS DATE) >= DATE '2025-01-05'
                            THEN 'post_cp'
                        ELSE 'pre_cp'
                    END AS pre_post_cp,

                    COUNT(*) AS trip_count,
                    AVG("{distance_col}") AS avg_trip_distance,
                    AVG({duration_seconds_expr}) AS avg_trip_duration_seconds

                FROM read_parquet('{source_path.as_posix()}')
                {where_clause}

                GROUP BY
                    dataset_type,
                    pickup_zone_id,
                    dropoff_zone_id,
                    date,
                    year,
                    month,
                    day_of_week,
                    hour,
                    temporal_bucket,
                    pre_post_cp
            )
            TO '{output_path.as_posix()}'
            (FORMAT PARQUET)
        """

        con = duckdb.connect()
        con.execute("SET memory_limit='3GB'")
        con.execute(f"SET temp_directory='{DUCKDB_TEMP_DIR.as_posix()}'")
        con.execute(query)
        con.close()

        print(
            f"[{i}/{total_jobs}] Wrote: {output_path.name}",
            flush=True
        )

        del query
        gc.collect()

else:

    print("All TLC mobility outputs already exist; generation skipped.")

Expected TLC mobility outputs: 118
Missing TLC mobility outputs: 0
Targeted TLC mobility rebuild outputs: 0
TLC mobility outputs to process this run: 0
All TLC mobility outputs already exist; generation skipped.


Before doing formal validation, let’s just look at what we created\. This gives us a quick gut\-check on the output grain, the fields we carried forward, and whether the aggregated mobility table looks like the object we intended to build\.

In [8]:
# ---------------------------------------------------------------------
# Inspect sample TLC mobility outputs by dataset
# ---------------------------------------------------------------------

for dataset_type in retained_tlc_datasets:

    sample_output_file = sorted(
        TLC_MOBILITY_OUTPUT_DIR.glob(
            f"{dataset_type}_tripdata_*_mobility.parquet"
        )
    )[0]

    print("=" * 80)
    print(dataset_type.upper())
    print("Sample file:", sample_output_file.name)
    print("=" * 80)

    sample_mobility_df = duckdb.sql(f"""
        SELECT *
        FROM read_parquet('{sample_output_file.as_posix()}')
        LIMIT 20
    """).df()

    display(sample_mobility_df)

YELLOW
Sample file: yellow_tripdata_2023-01_mobility.parquet


,dataset_type,pickup_zone_id,dropoff_zone_id,date,year,month,day_of_week,hour,temporal_bucket,pre_post_cp,trip_count,avg_trip_distance,avg_trip_duration_seconds
0,yellow,148,232,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,1,1.080000,413.000000
1,yellow,164,229,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,6,1.743333,651.666667
2,yellow,246,61,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,1,9.420000,2719.000000
3,yellow,263,142,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,3,1.970000,588.666667
4,yellow,229,137,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,9,1.053333,286.666667
5,yellow,132,225,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,4,10.742500,1871.750000
6,yellow,164,79,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,3,1.833333,693.666667
7,yellow,234,144,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,3,1.886667,666.000000
8,yellow,232,255,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,1,2.840000,869.000000
9,yellow,137,25,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,1,6.000000,843.000000


GREEN
Sample file: green_tripdata_2023-01_mobility.parquet


,dataset_type,pickup_zone_id,dropoff_zone_id,date,year,month,day_of_week,hour,temporal_bucket,pre_post_cp,trip_count,avg_trip_distance,avg_trip_duration_seconds
0,green,7,260,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,2,1.600000,515.000000
1,green,7,223,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,2,1.000000,411.000000
2,green,226,129,2023-01-01,2023,1,0,1,weekend_overnight,pre_cp,1,3.590000,2060.000000
3,green,256,61,2023-01-01,2023,1,0,3,weekend_overnight,pre_cp,1,3.020000,1042.000000
4,green,179,179,2023-01-01,2023,1,0,3,weekend_overnight,pre_cp,1,0.000000,11.000000
5,green,260,260,2023-01-01,2023,1,0,4,weekend_overnight,pre_cp,1,0.000000,4.000000
6,green,264,264,2023-01-01,2023,1,0,5,weekend_overnight,pre_cp,1,0.000000,2.000000
7,green,41,24,2023-01-01,2023,1,0,10,weekend_midday,pre_cp,1,0.550000,168.000000
8,green,74,244,2023-01-01,2023,1,0,10,weekend_midday,pre_cp,1,3.240000,622.000000
9,green,75,170,2023-01-01,2023,1,0,11,weekend_midday,pre_cp,1,3.900000,1130.000000


FHVHV
Sample file: fhvhv_tripdata_2023-01_mobility.parquet


,dataset_type,pickup_zone_id,dropoff_zone_id,date,year,month,day_of_week,hour,temporal_bucket,pre_post_cp,trip_count,avg_trip_distance,avg_trip_duration_seconds
0,fhvhv,19,21,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,1,25.010000,3256.000000
1,fhvhv,90,65,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,3,4.077333,1291.333333
2,fhvhv,114,165,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,1,9.580000,1468.000000
3,fhvhv,140,141,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,4,1.121750,456.500000
4,fhvhv,25,80,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,8,4.649375,935.875000
5,fhvhv,145,152,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,1,12.500000,1712.000000
6,fhvhv,144,228,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,3,7.330333,1060.666667
7,fhvhv,164,13,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,4,4.780000,1183.250000
8,fhvhv,262,74,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,3,2.540333,561.000000
9,fhvhv,229,259,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,1,14.310000,1793.000000


Findings\. The sample outputs look exactly like what we were hoping to create\. Rather than carrying hundreds of millions of individual trip records through the remainder of the pipeline, we now have compact mobility tables where each row represents an hourly origin\-destination mobility state\. The aggregation appears to be working correctly, with trip counts, average distances, and average trip durations being calculated for each Pickup Taxi Zone × Dropoff Taxi Zone × Date × Hour combination\.

Just as importantly, the process successfully standardized the retained Yellow Taxi, Green Taxi, and FHVHV datasets into a common structure\. Despite originating from different TLC systems, all three datasets now share the same spatial, temporal, and mobility fields, which will make downstream integration and analysis much simpler\. We also intentionally dropped many of the trip\-level operational and economic fields, such as fares, tolls, surcharges, and payment information\. While those fields may be useful for economic analyses, they are not central to the mobility\-focused goals of this project\. The resulting tables retain the information needed for clustering, anomaly detection, mobility regime analysis, and policy evaluation while dramatically reducing the amount of data that must be carried forward\.


Now that we have sample outputs, let’s quantify the practical payoff of aggregating early\. Instead of carrying the full trip\-level parquet files forward, we now have compact mobility\-state tables\. This check compares the raw source file sizes against the post\-aggregation outputs so we can see how much data volume we removed before downstream modeling\.

In [9]:
# ---------------------------------------------------------------------
# Compare raw TLC file sizes to mobility table outputs
# ---------------------------------------------------------------------

size_comparison_records = []

for dataset_type in retained_tlc_datasets:

    source_files = sorted(
        TLC_DATA_DIR.glob(f"{dataset_type}_tripdata_*.parquet")
    )

    output_files = sorted(
        TLC_MOBILITY_OUTPUT_DIR.glob(
            f"{dataset_type}_tripdata_*_mobility.parquet"
        )
    )

    source_size_mb = sum(
        path.stat().st_size / (1024 * 1024)
        for path in source_files
    )

    output_size_mb = sum(
        path.stat().st_size / (1024 * 1024)
        for path in output_files
    )

    size_comparison_records.append({
        "dataset_type": dataset_type,
        "source_files": len(source_files),
        "output_files": len(output_files),
        "source_size_mb": source_size_mb,
        "output_size_mb": output_size_mb,
        "size_reduction_mb": source_size_mb - output_size_mb,
        "size_reduction_pct": (
            1 - (output_size_mb / source_size_mb)
            if source_size_mb > 0 else None
        ),
        "compression_ratio": (
            source_size_mb / output_size_mb
            if output_size_mb > 0 else None
        ),
    })

size_comparison_df = pd.DataFrame(size_comparison_records)

size_comparison_df

,dataset_type,source_files,output_files,source_size_mb,output_size_mb,size_reduction_mb,size_reduction_pct,compression_ratio
0,yellow,40,40,2302.932751,497.774319,1805.158432,0.783852,4.626460
1,green,39,39,49.605245,13.291771,36.313474,0.732049,3.732027
2,fhvhv,39,39,18187.889832,3094.990265,15092.899568,0.829832,5.876558


Findings\. The aggregation step produced a substantial reduction in data volume across all retained TLC datasets\. Yellow Taxi shrank from approximately 2\.3 GB to 478 MB, Green Taxi from 50 MB to 12 MB, and FHVHV from 18\.2 GB to 3\.0 GB\. Overall, the resulting mobility tables are roughly 4–6 times smaller than the original trip\-level datasets while preserving the spatial, temporal, and mobility characteristics needed for downstream analysis\. These results validate the decision to aggregate the TLC data during standardization rather than carrying nearly one billion trip records forward into later stages of the pipeline\.

## 1\.2\.3\.2 TLC Temporal Validation

The mobility tables are built around a small set of derived temporal dimensions, including date, year, month, day\_of\_week, hour, temporal\_bucket, and pre\_post\_cp\. These fields drive nearly all of the downstream analysis, so it's worth taking a few minutes to confirm that they were populated correctly during the aggregation process\. Here we'll check for missing values, confirm that the expected temporal categories are present, and review the resulting distributions to make sure the retained TLC datasets provide complete and consistent temporal coverage across the study period\.

## Missingness and count validation of temporal fields

In [10]:
# ---------------------------------------------------------------------
# Validate TLC temporal fields
# ---------------------------------------------------------------------


RUN_TLC_TEMPORAL_FIELD_VALIDATION = False

tlc_mobility_files = sorted(
    TLC_MOBILITY_OUTPUT_DIR.glob("*_mobility.parquet")
)

temporal_fields = [
    "date",
    "year",
    "month",
    "day_of_week",
    "hour",
    "temporal_bucket",
    "pre_post_cp",
]

if (
    RUN_TLC_TEMPORAL_FIELD_VALIDATION
    or not TLC_TEMPORAL_VALIDATION_MANIFEST_PATH.exists()
):

    temporal_validation_records = []

    for field in temporal_fields:

        query = f"""
            SELECT
                '{field}' AS field,
                SUM(CASE WHEN "{field}" IS NULL THEN trip_count ELSE 0 END) AS missing_trip_count,
                SUM(trip_count) AS total_trip_count,
                COUNT(DISTINCT "{field}") AS unique_values
            FROM read_parquet(
                '{TLC_MOBILITY_OUTPUT_DIR.as_posix()}/*_mobility.parquet'
            )
        """

        result = duckdb.sql(query).df().iloc[0].to_dict()

        result["missing_rate"] = (
            result["missing_trip_count"] / result["total_trip_count"]
            if result["total_trip_count"] > 0
            else None
        )

        temporal_validation_records.append(result)

    tlc_temporal_validation_df = pd.DataFrame(
        temporal_validation_records
    )

    tlc_temporal_validation_df.to_csv(
        TLC_TEMPORAL_VALIDATION_MANIFEST_PATH,
        index=False
    )

    print(
        "Wrote TLC temporal validation manifest:",
        TLC_TEMPORAL_VALIDATION_MANIFEST_PATH
    )

else:

    tlc_temporal_validation_df = pd.read_csv(
        TLC_TEMPORAL_VALIDATION_MANIFEST_PATH
    )

    print(
        "Loaded existing TLC temporal validation manifest:",
        TLC_TEMPORAL_VALIDATION_MANIFEST_PATH
    )

tlc_temporal_validation_df[
    [
        "field",
        "missing_trip_count",
        "missing_rate",
        "unique_values",
    ]
]

Loaded existing TLC temporal validation manifest: pipeline_data/1.2.3.intermediate_tables/tlc_temporal_field_validation_manifest.csv


,field,missing_trip_count,missing_rate,unique_values
0,date,0.0,0.0,1228
1,year,0.0,0.0,12
2,month,0.0,0.0,12
3,day_of_week,0.0,0.0,7
4,hour,0.0,0.0,24
5,temporal_bucket,0.0,0.0,10
6,pre_post_cp,0.0,0.0,2


Findings\. The temporal validation results look clean\. None of the derived temporal fields contain missing values, which confirms that the aggregation process successfully assigned a valid timestamp\-derived representation to every retained TLC trip\. The resulting mobility tables provide complete coverage across all expected months, days of the week, hours, temporal buckets, and congestion\-pricing periods\.

The only result that warrants a closer look is the year field, which reports 12 unique values rather than the expected four years spanning 2023–2026\. This suggests that one or more files may contain unexpected year values or that a validation issue exists in the aggregation process\. We'll investigate that anomaly before closing out the TLC temporal QA\. Aside from that item, the temporal fields appear complete and ready for downstream mobility\-state analysis\.

In [11]:
# ---------------------------------------------------------------------
# Inspect year values
# ---------------------------------------------------------------------

duckdb.sql(f"""
    SELECT
        year,
        COUNT(*) AS mobility_states,
        SUM(trip_count) AS trips
    FROM read_parquet(
        '{TLC_MOBILITY_OUTPUT_DIR.as_posix()}/*_mobility.parquet'
    )
    GROUP BY 1
    ORDER BY 1
""").df()

,year,mobility_states,trips
0,2023,97520823,271586435.0
1,2024,100066279,281299802.0
2,2025,104011782,292903199.0
3,2026,26770577,77874044.0


Findings\. The temporal validation identified a very small number of records with year values outside the project's 2023–2026 study period\. These records account for fewer than 200 trips out of more than 923 million retained trips and are therefore not expected to have any meaningful impact on downstream analysis\. As a result, they were retained and no additional cleaning was performed\.

### Temporal bucket distributions

In [12]:
# ---------------------------------------------------------------------
# Temporal bucket distribution
# ---------------------------------------------------------------------

tlc_temporal_bucket_distribution = duckdb.sql(f"""
    SELECT
        temporal_bucket,
        SUM(trip_count) AS trip_count,
        SUM(trip_count) * 1.0
            / SUM(SUM(trip_count)) OVER () AS trip_share
    FROM read_parquet(
        '{TLC_MOBILITY_OUTPUT_DIR.as_posix()}/*_mobility.parquet'
    )
    GROUP BY 1
    ORDER BY 1
""").df()

tlc_temporal_bucket_distribution

,temporal_bucket,trip_count,trip_share
0,weekday_am_peak,118267822.0,0.128042
1,weekday_evening,134748392.0,0.145885
2,weekday_midday,178695784.0,0.193464
3,weekday_overnight,58219622.0,0.063031
4,weekday_pm_peak,151351907.0,0.163860
5,weekend_am_peak,28945847.0,0.031338
6,weekend_evening,58336538.0,0.063158
7,weekend_midday,79091972.0,0.085629
8,weekend_overnight,54351267.0,0.058843
9,weekend_pm_peak,61654329.0,0.066750


Findings\. The temporal bucket distribution looks reasonable and aligns with what we'd expect from a citywide mobility dataset\. Most trips occur during weekdays, with weekday midday, weekday overnight, and weekday PM peak periods accounting for the largest shares of activity\. Weekend travel patterns are also well represented, although at lower volumes than weekdays\. More importantly, all eight temporal buckets are populated, confirming that the aggregation process successfully captured mobility activity across the full range of expected day and time combinations\. This balanced coverage should provide a solid foundation for downstream clustering, anomaly detection, and mobility regime analysis\.

In [13]:
# ---------------------------------------------------------------------
# Pre/post congestion pricing distribution
# ---------------------------------------------------------------------

tlc_pre_post_distribution = duckdb.sql(f"""
    SELECT
        pre_post_cp,
        SUM(trip_count) AS trip_count,
        SUM(trip_count) * 1.0
            / SUM(SUM(trip_count)) OVER () AS trip_share
    FROM read_parquet(
        '{TLC_MOBILITY_OUTPUT_DIR.as_posix()}/*_mobility.parquet'
    )
    GROUP BY 1
    ORDER BY 1
""").df()

tlc_pre_post_distribution

,pre_post_cp,trip_count,trip_share
0,post_cp,367929936.0,0.398338
1,pre_cp,555733544.0,0.601662


Findings\. Both congestion\-pricing periods are well represented in the resulting mobility tables\. Approximately 60% of trips fall within the pre\-congestion pricing period and roughly 40% fall within the post\-congestion pricing period\. This distribution is consistent with the study timeframe and confirms that the pre\_post\_cp field was assigned correctly during aggregation\. The strong representation of both periods should provide sufficient coverage for future comparative analyses examining how congestion pricing may have influenced mobility patterns across the city\.

## 1\.2\.3\.3 Spatial Validation

The mobility tables are built around pickup and dropoff Taxi Zones, so the next step is to confirm that the aggregation preserved the expected spatial coverage\. Here we'll examine pickup and dropoff zone coverage and verify that the resulting mobility tables continue to represent activity across the full Taxi Zone system\.

### Coverage validation

First, let's check whether  zone fields are complete and broad enough to support citywide analysis\. Here we validate missingness and unique zone coverage for both pickup and dropoff zones, then attach the Taxi Zone reference layer so the results are readable by borough and neighborhood name instead of raw zone IDs\.

In [14]:
# ---------------------------------------------------------------------
# Load Taxi Zone reference layer
# ---------------------------------------------------------------------

TAXI_ZONES_PATH = PIPELINE_DATA_DIR / "1.1.6.nyc_taxi_zones.parquet"

taxi_zones_lookup = (
    pd.read_parquet(TAXI_ZONES_PATH)
    [["location_id", "borough", "zone"]]
    .drop_duplicates()
    .rename(columns={
        "location_id": "taxi_zone_id",
        "borough": "taxi_zone_borough",
        "zone": "taxi_zone_name",
    })
)

taxi_zones_lookup.head()

,taxi_zone_id,taxi_zone_borough,taxi_zone_name
0,1,EWR,Newark Airport
1,2,Queens,Jamaica Bay
2,3,Bronx,Allerton/Pelham Gardens
3,4,Manhattan,Alphabet City
4,5,Staten Island,Arden Heights


In [15]:
# ---------------------------------------------------------------------
# Test 1: Pickup/dropoff Taxi Zone coverage
# ---------------------------------------------------------------------

tlc_zone_coverage = duckdb.sql(f"""
    SELECT
        COUNT(*) AS mobility_states,
        SUM(trip_count) AS trips,

        SUM(
            CASE
                WHEN pickup_zone_id IS NULL
                THEN trip_count ELSE 0
            END
        ) AS missing_pickup_zone_trips,

        SUM(
            CASE
                WHEN dropoff_zone_id IS NULL
                THEN trip_count ELSE 0
            END
        ) AS missing_dropoff_zone_trips,

        COUNT(DISTINCT pickup_zone_id) AS unique_pickup_zones,
        COUNT(DISTINCT dropoff_zone_id) AS unique_dropoff_zones
    FROM read_parquet(
        '{TLC_MOBILITY_OUTPUT_DIR.as_posix()}/*_mobility.parquet'
    )
""").df()

tlc_zone_coverage["missing_pickup_zone_rate"] = (
    tlc_zone_coverage["missing_pickup_zone_trips"]
    / tlc_zone_coverage["trips"]
)

tlc_zone_coverage["missing_dropoff_zone_rate"] = (
    tlc_zone_coverage["missing_dropoff_zone_trips"]
    / tlc_zone_coverage["trips"]
)

tlc_zone_coverage

,mobility_states,trips,missing_pickup_zone_trips,missing_dropoff_zone_trips,unique_pickup_zones,unique_dropoff_zones,missing_pickup_zone_rate,missing_dropoff_zone_rate
0,328369461,923663480.0,0.0,0.0,263,264,0.0,0.0


Findings\. The mobility tables retain essentially complete Taxi Zone coverage, with no missing pickup or dropoff zones introduced during aggregation\. Coverage remains consistent with the source\-level TLC QA, suggesting that the transition from trip\-level records to mobility states preserved the underlying spatial structure of the data\.

In [16]:
# ---------------------------------------------------------------------
# Test 1b: Pickup/dropoff Taxi Zone coverage by dataset
# ---------------------------------------------------------------------

tlc_zone_coverage_by_dataset = duckdb.sql(f"""
    SELECT
        dataset_type,
        COUNT(*) AS mobility_states,
        SUM(trip_count) AS trips,

        SUM(
            CASE
                WHEN pickup_zone_id IS NULL
                THEN trip_count ELSE 0
            END
        ) AS missing_pickup_zone_trips,

        SUM(
            CASE
                WHEN dropoff_zone_id IS NULL
                THEN trip_count ELSE 0
            END
        ) AS missing_dropoff_zone_trips,

        COUNT(DISTINCT pickup_zone_id) AS unique_pickup_zones,
        COUNT(DISTINCT dropoff_zone_id) AS unique_dropoff_zones
    FROM read_parquet(
        '{TLC_MOBILITY_OUTPUT_DIR.as_posix()}/*_mobility.parquet'
    )
    GROUP BY 1
    ORDER BY 1
""").df()

tlc_zone_coverage_by_dataset["missing_pickup_zone_rate"] = (
    tlc_zone_coverage_by_dataset["missing_pickup_zone_trips"]
    / tlc_zone_coverage_by_dataset["trips"]
)

tlc_zone_coverage_by_dataset["missing_dropoff_zone_rate"] = (
    tlc_zone_coverage_by_dataset["missing_dropoff_zone_trips"]
    / tlc_zone_coverage_by_dataset["trips"]
)

tlc_zone_coverage_by_dataset

,dataset_type,mobility_states,trips,missing_pickup_zone_trips,missing_dropoff_zone_trips,unique_pickup_zones,unique_dropoff_zones,missing_pickup_zone_rate,missing_dropoff_zone_rate
0,fhvhv,272691530,778424569.0,0.0,0.0,263,264,0.0,0.0
1,green,1690028,2159911.0,0.0,0.0,258,260,0.0,0.0
2,yellow,53987903,143079000.0,0.0,0.0,263,263,0.0,0.0


Findings\. Spatial coverage remains strong across all three retained TLC datasets\. Yellow Taxi and FHVHV cover essentially the entire Taxi Zone system, while Green Taxi covers slightly fewer zones, which is expected given its smaller footprint\. One item worth following up on is the additional dropoff zone observed in the aggregated data, which appears to persist from the source datasets and will be investigated further during harmonization QA\.

In [17]:
# ---------------------------------------------------------------------
# Identify pickup zone IDs not found in the Taxi Zone reference layer
# ---------------------------------------------------------------------

official_taxi_zone_ids = set(
    taxi_zones_lookup["taxi_zone_id"]
)

pickup_zone_usage = duckdb.sql(f"""
    SELECT
        dataset_type,
        pickup_zone_id AS zone_id,
        SUM(trip_count) AS trips
    FROM read_parquet(
        '{TLC_MOBILITY_OUTPUT_DIR.as_posix()}/*_mobility.parquet'
    )
    GROUP BY 1, 2
""").df()

unmatched_pickup_zone_usage = (
    pickup_zone_usage[
        ~pickup_zone_usage["zone_id"].isin(official_taxi_zone_ids)
    ]
    .sort_values(["zone_id", "dataset_type"])
)

unmatched_pickup_zone_usage

,dataset_type,zone_id,trips
658,fhvhv,57,177310.0
304,green,57,400.0
589,yellow,57,1716.0
261,fhvhv,105,133.0
195,yellow,105,24.0
76,fhvhv,264,5.0
460,green,264,4576.0
342,yellow,264,577013.0
243,fhvhv,265,33835.0
687,green,265,1819.0


Findings\. Spatial validation identified four pickup zone identifiers that do not currently resolve against the staged Spatial Reference Layer: 57, 105, 264, and 265\. Zone IDs 57, 264, and 265 appear across all three TLC datasets, while Zone 105 appears only in Yellow Taxi and FHVHV\. The fact that these identifiers appear consistently across multiple TLC modalities suggests they are legitimate location codes rather than isolated data\-quality issues introduced during aggregation\.

In [18]:
# ---------------------------------------------------------------------
# Summarize unmatched pickup zone IDs
# ---------------------------------------------------------------------

total_pickup_trips = pickup_zone_usage["trips"].sum()

unmatched_pickup_zone_summary = (
    unmatched_pickup_zone_usage
    .groupby("zone_id", dropna=False)
    .agg(
        trips=("trips", "sum"),
        datasets=("dataset_type", lambda x: ", ".join(sorted(x.unique()))),
    )
    .reset_index()
    .sort_values("zone_id")
)

unmatched_pickup_zone_summary["trip_share"] = (
    unmatched_pickup_zone_summary["trips"]
    / total_pickup_trips
)

unmatched_pickup_zone_summary

,zone_id,trips,datasets,trip_share
0,57,179426.0,"fhvhv, green, yellow",1.942547e-04
1,105,157.0,"fhvhv, yellow",1.699753e-07
2,264,581594.0,"fhvhv, green, yellow",6.296601e-04
3,265,118288.0,"fhvhv, green, yellow",1.280640e-04


Findings\. Although four pickup zone identifiers did not resolve against the current Spatial Reference Layer, their overall impact is minimal\. Together, these zones account for approximately 880 thousand trips, or less than 0\.1% of all retained TLC trips\. Zone 264 contributes the largest share of unmatched activity, followed by Zones 57 and 265, while Zone 105 appears only rarely\. These results indicate that spatial coverage remains effectively complete, with only a very small fraction of TLC activity requiring reconciliation with the Spatial Reference Layer\.

## 1\.2\.3\.4 Final TLC Pickup\-Temporal Contract

The earlier monthly mobility tables preserve the origin\-destination detail we need, but they still carry Yellow Taxi, Green Taxi, and FHVHV as separate source datasets\. Downstream integration work showed that Yellow and Green should become one Taxi modality, while FHVHV should remain separate\. We apply that decision here while pickup\-dropoff detail is still available, then write the compact pickup\-zone temporal contract needed by harmonization and panel integration\. This lets Taxi trip counts, average trip distance, average trip duration, average speed, and destination diversity be computed from valid aggregate numerators and denominators without materializing a large merged OD table by default\.

This final contract also carries forward the project\-wide temporal ordering fields introduced in \`1\.2\.1\` and adds native TLC spatial provenance metadata consistent with \`1\.2\.2\`\. The required output is written to a separate \`1\.2\.3\.final\_tlc\_contract\` directory as monthly parquet chunks so the run is restartable, progress is visible, and \`1\.2\.4\` and \`1\.3\.1\` can migrate to the cleaned Taxi/FHVHV contract\. A merged OD audit output remains available behind a toggle, but it is not part of the default pipeline\.

In [19]:
# ---------------------------------------------------------------------
# Build final Taxi/FHVHV TLC pickup-temporal contract
# ---------------------------------------------------------------------

FINAL_TLC_CONTRACT_BUILD_STRATEGY = "service_month_filtered_monthly_pickup_temporal_v3"
STUDY_START_DATE = pd.Timestamp("2023-01-01")
STUDY_END_DATE = pd.Timestamp("2026-03-31")


def parse_tlc_service_month(path):
    return (
        path.name
        .split("_tripdata_")[1]
        .replace("_mobility.parquet", "")
    )


legacy_tlc_mobility_files = sorted(
    TLC_MOBILITY_OUTPUT_DIR.glob("*_tripdata_*_mobility.parquet")
)

study_months = pd.date_range(
    STUDY_START_DATE,
    STUDY_END_DATE,
    freq="MS"
)

final_contract_metadata = {}

if TLC_FINAL_MODALITY_CONTRACT_PATH.exists():
    with open(TLC_FINAL_MODALITY_CONTRACT_PATH, "r") as f:
        final_contract_metadata = json.load(f)

existing_pickup_chunks = sorted(
    TLC_PICKUP_TEMPORAL_FINAL_OUTPUT_DIR.glob("*.parquet")
)

contract_outputs_current = (
    final_contract_metadata.get("build_strategy") == FINAL_TLC_CONTRACT_BUILD_STRATEGY
    and len(existing_pickup_chunks) == len(study_months)
)

if (
    REBUILD_FINAL_TLC_MODALITY_CONTRACT
    or not contract_outputs_current
):

    print("Building final TLC Taxi/FHVHV pickup-temporal contract outputs...", flush=True)
    print("Source monthly OD files:", len(legacy_tlc_mobility_files), flush=True)
    print("Monthly chunks:", len(study_months), flush=True)

    if TLC_PICKUP_TEMPORAL_FINAL_OUTPUT_DIR.exists():
        shutil.rmtree(TLC_PICKUP_TEMPORAL_FINAL_OUTPUT_DIR)

    TLC_PICKUP_TEMPORAL_FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    if WRITE_FINAL_TLC_OD_AUDIT_OUTPUT:
        if TLC_OD_FINAL_OUTPUT_DIR.exists():
            shutil.rmtree(TLC_OD_FINAL_OUTPUT_DIR)
        TLC_OD_FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    build_start_time = time.time()
    monthly_contract_records = []

    for month_index, month_start in enumerate(study_months, start=1):

        month_end = month_start + pd.offsets.MonthBegin(1)
        service_month = month_start.strftime("%Y-%m")

        month_source_files = [
            path for path in legacy_tlc_mobility_files
            if parse_tlc_service_month(path) == service_month
        ]

        if not month_source_files:
            print(
                f"[{month_index}/{len(study_months)}] No TLC mobility files found for {service_month}; skipping.",
                flush=True
            )
            continue

        source_file_sql_list = ", ".join(
            f"'{path.as_posix()}'" for path in month_source_files
        )

        pickup_output_path = (
            TLC_PICKUP_TEMPORAL_FINAL_OUTPUT_DIR /
            f"tlc_pickup_temporal_{service_month}.parquet"
        )

        od_audit_output_path = (
            TLC_OD_FINAL_OUTPUT_DIR /
            f"tlc_od_{service_month}.parquet"
        )

        month_start_time = time.time()

        print(
            f"[{month_index}/{len(study_months)}] Building {service_month} "
            f"from {len(month_source_files)} service-month files...",
            flush=True
        )
        print(
            "    Source files: " + ", ".join(path.name for path in month_source_files),
            flush=True
        )

        pickup_query = f"""
            COPY (
                WITH od_rows AS (
                    SELECT
                        CASE
                            WHEN dataset_type IN ('yellow', 'green') THEN 'taxi'
                            WHEN dataset_type = 'fhvhv' THEN 'fhvhv'
                            ELSE dataset_type
                        END AS modality,
                        dataset_type,
                        pickup_zone_id AS taxi_zone_id,
                        dropoff_zone_id,
                        CAST(date AS DATE) AS date,
                        year,
                        month,
                        day_of_week,
                        temporal_bucket,
                        pre_post_cp,
                        trip_count,
                        avg_trip_distance,
                        avg_trip_duration_seconds
                    FROM read_parquet([{source_file_sql_list}])
                    WHERE CAST(date AS DATE) >= DATE '{month_start.date()}'
                      AND CAST(date AS DATE) < DATE '{month_end.date()}'
                      AND CAST(date AS DATE) BETWEEN DATE '{STUDY_START_DATE.date()}'
                          AND DATE '{STUDY_END_DATE.date()}'
                ),
                pickup_temporal AS (
                    SELECT
                        modality,
                        taxi_zone_id,
                        date,
                        year,
                        month,
                        day_of_week,
                        temporal_bucket,
                        pre_post_cp,

                        CASE
                            WHEN temporal_bucket LIKE '%overnight' THEN 'overnight'
                            WHEN temporal_bucket LIKE '%am_peak' THEN 'am_peak'
                            WHEN temporal_bucket LIKE '%midday' THEN 'midday'
                            WHEN temporal_bucket LIKE '%pm_peak' THEN 'pm_peak'
                            WHEN temporal_bucket LIKE '%evening' THEN 'evening'
                            ELSE NULL
                        END AS daypart,

                        CASE
                            WHEN temporal_bucket LIKE '%overnight' THEN 1
                            WHEN temporal_bucket LIKE '%am_peak' THEN 2
                            WHEN temporal_bucket LIKE '%midday' THEN 3
                            WHEN temporal_bucket LIKE '%pm_peak' THEN 4
                            WHEN temporal_bucket LIKE '%evening' THEN 5
                            ELSE NULL
                        END AS daypart_order,

                        CASE temporal_bucket
                            WHEN 'weekday_overnight' THEN 1
                            WHEN 'weekday_am_peak' THEN 2
                            WHEN 'weekday_midday' THEN 3
                            WHEN 'weekday_pm_peak' THEN 4
                            WHEN 'weekday_evening' THEN 5
                            WHEN 'weekend_overnight' THEN 6
                            WHEN 'weekend_am_peak' THEN 7
                            WHEN 'weekend_midday' THEN 8
                            WHEN 'weekend_pm_peak' THEN 9
                            WHEN 'weekend_evening' THEN 10
                            ELSE NULL
                        END AS temporal_bucket_order,

                        SUM(trip_count) AS trip_count,
                        SUM(avg_trip_distance * trip_count) AS trip_distance_sum,
                        SUM(avg_trip_duration_seconds * trip_count) AS trip_duration_seconds_sum,
                        SUM(CASE WHEN avg_trip_distance IS NOT NULL THEN trip_count ELSE 0 END)
                            AS valid_trip_count_for_distance,
                        SUM(CASE WHEN avg_trip_duration_seconds IS NOT NULL THEN trip_count ELSE 0 END)
                            AS valid_trip_count_for_duration,
                        COUNT(DISTINCT dropoff_zone_id) AS distinct_dropoff_zone_count,
                        COUNT(*) AS source_od_row_count,

                        SUM(CASE WHEN dataset_type = 'yellow' THEN trip_count ELSE 0 END)
                            AS yellow_trip_count,
                        SUM(CASE WHEN dataset_type = 'green' THEN trip_count ELSE 0 END)
                            AS green_trip_count,
                        SUM(CASE WHEN dataset_type = 'fhvhv' THEN trip_count ELSE 0 END)
                            AS fhvhv_source_trip_count,

                        SUM(CASE WHEN dataset_type = 'yellow' THEN 1 ELSE 0 END)
                            AS yellow_source_od_row_count,
                        SUM(CASE WHEN dataset_type = 'green' THEN 1 ELSE 0 END)
                            AS green_source_od_row_count,
                        SUM(CASE WHEN dataset_type = 'fhvhv' THEN 1 ELSE 0 END)
                            AS fhvhv_source_od_row_count
                    FROM od_rows
                    GROUP BY
                        modality,
                        taxi_zone_id,
                        date,
                        year,
                        month,
                        day_of_week,
                        temporal_bucket,
                        pre_post_cp
                )
                SELECT
                    modality,
                    taxi_zone_id,
                    date,
                    year,
                    month,
                    day_of_week,
                    daypart,
                    daypart_order,
                    temporal_bucket,
                    temporal_bucket_order,
                    (
                        DATE_DIFF('day', DATE '{STUDY_START_DATE.date()}', date) * 5
                        + daypart_order
                    ) AS observation_sequence_id,
                    pre_post_cp,
                    trip_count,
                    trip_distance_sum,
                    trip_duration_seconds_sum,
                    valid_trip_count_for_distance,
                    valid_trip_count_for_duration,
                    CASE
                        WHEN valid_trip_count_for_distance > 0
                        THEN trip_distance_sum / valid_trip_count_for_distance
                        ELSE NULL
                    END AS avg_trip_distance,
                    CASE
                        WHEN valid_trip_count_for_duration > 0
                        THEN trip_duration_seconds_sum / valid_trip_count_for_duration
                        ELSE NULL
                    END AS avg_trip_duration_seconds,
                    CASE
                        WHEN trip_duration_seconds_sum > 0
                        THEN trip_distance_sum / (trip_duration_seconds_sum / 3600.0)
                        ELSE NULL
                    END AS avg_trip_speed_mph,
                    distinct_dropoff_zone_count,
                    source_od_row_count,
                    yellow_trip_count,
                    green_trip_count,
                    fhvhv_source_trip_count,
                    yellow_source_od_row_count,
                    green_source_od_row_count,
                    fhvhv_source_od_row_count,
                    'native_tlc_taxi_zone_id' AS spatial_assignment_method,
                    'tlc_source_pickup_location_id' AS spatial_assignment_source,
                    'native' AS spatial_assignment_confidence,
                    'taxi_zone' AS canonical_geography
                FROM pickup_temporal
                ORDER BY
                    modality,
                    taxi_zone_id,
                    date,
                    temporal_bucket_order
            )
            TO '{pickup_output_path.as_posix()}'
            (FORMAT PARQUET)
        """

        con = duckdb.connect()
        con.execute("PRAGMA threads=1")
        con.execute("PRAGMA memory_limit='8GB'")
        con.execute(f"SET temp_directory='{DUCKDB_TEMP_DIR.as_posix()}'")
        con.execute(pickup_query)

        if WRITE_FINAL_TLC_OD_AUDIT_OUTPUT:
            od_audit_query = f"""
                COPY (
                    SELECT
                        CASE
                            WHEN dataset_type IN ('yellow', 'green') THEN 'taxi'
                            WHEN dataset_type = 'fhvhv' THEN 'fhvhv'
                            ELSE dataset_type
                        END AS modality,
                        dataset_type,
                        pickup_zone_id AS taxi_zone_id,
                        dropoff_zone_id,
                        CAST(date AS DATE) AS date,
                        year,
                        month,
                        day_of_week,
                        temporal_bucket,
                        pre_post_cp,
                        trip_count,
                        avg_trip_distance,
                        avg_trip_duration_seconds,
                        'native_tlc_taxi_zone_id' AS spatial_assignment_method,
                        'tlc_source_pickup_location_id' AS spatial_assignment_source,
                        'native' AS spatial_assignment_confidence,
                        'taxi_zone' AS canonical_geography
                    FROM read_parquet([{source_file_sql_list}])
                    WHERE CAST(date AS DATE) >= DATE '{month_start.date()}'
                      AND CAST(date AS DATE) < DATE '{month_end.date()}'
                      AND CAST(date AS DATE) BETWEEN DATE '{STUDY_START_DATE.date()}'
                          AND DATE '{STUDY_END_DATE.date()}'
                    ORDER BY
                        modality,
                        dataset_type,
                        taxi_zone_id,
                        dropoff_zone_id,
                        date,
                        temporal_bucket
                )
                TO '{od_audit_output_path.as_posix()}'
                (FORMAT PARQUET)
            """
            con.execute(od_audit_query)

        con.close()

        elapsed_minutes = (time.time() - month_start_time) / 60
        monthly_contract_records.append({
            "service_month": service_month,
            "source_files": len(month_source_files),
            "pickup_output": pickup_output_path.as_posix(),
            "od_audit_output": (
                od_audit_output_path.as_posix()
                if WRITE_FINAL_TLC_OD_AUDIT_OUTPUT
                else None
            ),
            "elapsed_minutes": round(elapsed_minutes, 2),
        })

        print(
            f"[{month_index}/{len(study_months)}] Wrote {pickup_output_path.name} "
            f"in {elapsed_minutes:.2f} minutes.",
            flush=True
        )

        del pickup_query
        gc.collect()

    contract_metadata = {
        "contract_name": "final_tlc_pickup_temporal_mobility",
        "build_strategy": FINAL_TLC_CONTRACT_BUILD_STRATEGY,
        "study_start_date": str(STUDY_START_DATE.date()),
        "study_end_date": str(STUDY_END_DATE.date()),
        "source_grain": "TLC service-month OD mobility outputs",
        "output_grain": "modality x pickup Taxi Zone x date x temporal_bucket",
        "modalities": ["taxi", "fhvhv"],
        "taxi_definition": "Yellow Taxi and Green Taxi combined upstream from service-month OD rows.",
        "fhvhv_definition": "FHVHV retained as its own modality.",
        "destination_diversity_definition": "COUNT(DISTINCT dropoff_zone_id) computed before pickup-temporal collapse.",
        "metric_definition": "Trip counts are summed directly; distance, duration, and speed are recomputed from summed numerators and valid denominators.",
        "spatial_provenance": "Native TLC Pickup Taxi Zone identifiers; not uncertainty weighted.",
        "write_od_audit_output": WRITE_FINAL_TLC_OD_AUDIT_OUTPUT,
        "monthly_chunks": monthly_contract_records,
        "total_elapsed_minutes": round((time.time() - build_start_time) / 60, 2),
    }

    with open(TLC_FINAL_MODALITY_CONTRACT_PATH, "w") as f:
        json.dump(contract_metadata, f, indent=2)

    print("Wrote final TLC contract metadata:", TLC_FINAL_MODALITY_CONTRACT_PATH)
    print(
        "Final TLC contract build completed in "
        f"{contract_metadata['total_elapsed_minutes']:.2f} minutes.",
        flush=True
    )

else:

    print(
        "Final TLC pickup-temporal contract outputs already exist for the current "
        "build strategy; build skipped."
    )

Final TLC pickup-temporal contract outputs already exist for the current build strategy; build skipped.


Before moving on, validate the new TLC pickup\-temporal contract directly\. This check confirms that the final outputs contain only the two approved downstream modalities, preserve retained valid trip volume from the monthly OD tables, stay inside the study period, populate temporal ordering and spatial provenance fields, and expose the grain needed by \`1\.2\.4\`\. Destination diversity is validated as a metric computed while \`dropoff\_zone\_id\` is still present in the source rows, not as a downstream sum of service\-level distinct counts\.

In [20]:
# ---------------------------------------------------------------------
# Validate final Taxi/FHVHV TLC pickup-temporal contract
# ---------------------------------------------------------------------

FINAL_TLC_CONTRACT_VALIDATION_STRATEGY = "service_month_filtered_monthly_pickup_temporal_validation_v4"

if TLC_FINAL_MODALITY_CONTRACT_PATH.exists():
    with open(TLC_FINAL_MODALITY_CONTRACT_PATH, "r") as f:
        final_contract_metadata = json.load(f)
else:
    final_contract_metadata = {}

current_contract_build_strategy = final_contract_metadata.get(
    "build_strategy",
    globals().get("FINAL_TLC_CONTRACT_BUILD_STRATEGY", "unknown")
)

cached_validation_strategy = None
cached_contract_build_strategy = None

if TLC_FINAL_CONTRACT_VALIDATION_PATH.exists():
    cached_validation_header_df = pd.read_csv(
        TLC_FINAL_CONTRACT_VALIDATION_PATH,
        nrows=1
    )

    if "validation_strategy" in cached_validation_header_df.columns:
        cached_validation_strategy = cached_validation_header_df["validation_strategy"].iloc[0]

    if "contract_build_strategy" in cached_validation_header_df.columns:
        cached_contract_build_strategy = cached_validation_header_df["contract_build_strategy"].iloc[0]

validation_cache_is_current = (
    TLC_FINAL_CONTRACT_VALIDATION_PATH.exists()
    and cached_validation_strategy == FINAL_TLC_CONTRACT_VALIDATION_STRATEGY
    and cached_contract_build_strategy == current_contract_build_strategy
)

if (
    RUN_FINAL_TLC_MODALITY_CONTRACT_VALIDATION
    or not validation_cache_is_current
):

    if TLC_FINAL_CONTRACT_VALIDATION_PATH.exists() and not validation_cache_is_current:
        print(
            "Cached final TLC contract validation is stale; rebuilding validation from current outputs.",
            flush=True
        )

    final_pickup_files = sorted(TLC_PICKUP_TEMPORAL_FINAL_OUTPUT_DIR.glob("*.parquet"))

    validation_source_files = []

    for path in sorted(TLC_MOBILITY_OUTPUT_DIR.glob("*_tripdata_*_mobility.parquet")):
        service_month = parse_tlc_service_month(path)
        service_month_start = pd.Timestamp(f"{service_month}-01")

        if STUDY_START_DATE <= service_month_start <= STUDY_END_DATE:
            validation_source_files.append(path)

    assert final_pickup_files, (
        "No final TLC pickup-temporal contract files found. "
        "Run the final contract build cell before validation."
    )

    assert validation_source_files, (
        "No TLC source mobility files found for final contract validation. "
        "Run the TLC mobility table generation cell before validation."
    )

    print(
        "Checking TLC monthly mobility outputs for service-month date leakage...",
        flush=True
    )

    service_month_leakage_records = []

    for file_index, source_path in enumerate(validation_source_files, start=1):
        service_month = parse_tlc_service_month(source_path)
        service_month_start = pd.Timestamp(f"{service_month}-01")
        next_service_month_start = service_month_start + pd.offsets.MonthBegin(1)

        leakage_df = duckdb.sql(f"""
            SELECT
                dataset_type,
                COUNT(*) AS out_of_service_month_rows,
                SUM(trip_count) AS out_of_service_month_trips,
                MIN(CAST(date AS DATE)) AS min_leaked_date,
                MAX(CAST(date AS DATE)) AS max_leaked_date
            FROM read_parquet('{source_path.as_posix()}')
            WHERE CAST(date AS DATE) BETWEEN DATE '{STUDY_START_DATE.date()}'
                AND DATE '{STUDY_END_DATE.date()}'
              AND (
                    CAST(date AS DATE) < DATE '{service_month_start.date()}'
                 OR CAST(date AS DATE) >= DATE '{next_service_month_start.date()}'
              )
            GROUP BY dataset_type
            HAVING COUNT(*) > 0
        """).df()

        if not leakage_df.empty:
            leakage_df.insert(0, "source_file", source_path.name)
            leakage_df.insert(1, "service_month", service_month)
            service_month_leakage_records.append(leakage_df)

    if service_month_leakage_records:
        tlc_service_month_leakage_df = pd.concat(
            service_month_leakage_records,
            ignore_index=True
        )
    else:
        tlc_service_month_leakage_df = pd.DataFrame(
            columns=[
                "source_file",
                "service_month",
                "dataset_type",
                "out_of_service_month_rows",
                "out_of_service_month_trips",
                "min_leaked_date",
                "max_leaked_date",
            ]
        )

    if not tlc_service_month_leakage_df.empty:
        print(
            "Monthly TLC mobility outputs still contain out-of-service-month rows. "
            "Rebuild the monthly TLC mobility tables with "
            "REBUILD_ALL_TLC_MOBILITY_TABLES_WITH_SERVICE_MONTH_FILTER = True, "
            "then rerun the final contract build and validation.",
            flush=True
        )
        display(tlc_service_month_leakage_df)

    assert tlc_service_month_leakage_df.empty, (
        "TLC monthly mobility outputs are not service-month clean. "
        "The final contract cannot be validated until these upstream files are rebuilt. "
        "See tlc_service_month_leakage_df for affected files."
    )

    final_pickup_file_sql_list = ", ".join(
        f"'{path.as_posix()}'" for path in final_pickup_files
    )

    validation_source_file_sql_list = ", ".join(
        f"'{path.as_posix()}'" for path in validation_source_files
    )

    print(
        "Validating final TLC pickup-temporal contract from "
        f"{len(validation_source_files)} study-period source files...",
        flush=True
    )

    validation_query = f"""
        WITH source_od AS (
            SELECT
                CASE
                    WHEN dataset_type IN ('yellow', 'green') THEN 'taxi'
                    WHEN dataset_type = 'fhvhv' THEN 'fhvhv'
                    ELSE dataset_type
                END AS modality,
                dataset_type,
                CAST(date AS DATE) AS date,
                trip_count
            FROM read_parquet([{validation_source_file_sql_list}])
            WHERE CAST(date AS DATE) BETWEEN DATE '{STUDY_START_DATE.date()}'
                AND DATE '{STUDY_END_DATE.date()}'
        ),
        source_summary AS (
            SELECT
                modality,
                SUM(trip_count) AS source_trip_count,
                SUM(CASE WHEN dataset_type = 'yellow' THEN trip_count ELSE 0 END)
                    AS source_yellow_trip_count,
                SUM(CASE WHEN dataset_type = 'green' THEN trip_count ELSE 0 END)
                    AS source_green_trip_count,
                SUM(CASE WHEN dataset_type = 'fhvhv' THEN trip_count ELSE 0 END)
                    AS source_fhvhv_trip_count,
                COUNT(*) AS source_od_row_count,
                SUM(CASE WHEN dataset_type = 'yellow' THEN 1 ELSE 0 END)
                    AS source_yellow_od_row_count,
                SUM(CASE WHEN dataset_type = 'green' THEN 1 ELSE 0 END)
                    AS source_green_od_row_count,
                SUM(CASE WHEN dataset_type = 'fhvhv' THEN 1 ELSE 0 END)
                    AS source_fhvhv_od_row_count,
                MIN(date) AS source_min_date,
                MAX(date) AS source_max_date
            FROM source_od
            GROUP BY modality
        ),
        pickup_summary AS (
            SELECT
                modality,
                SUM(trip_count) AS pickup_trip_count,
                SUM(yellow_trip_count) AS pickup_yellow_trip_count,
                SUM(green_trip_count) AS pickup_green_trip_count,
                SUM(fhvhv_source_trip_count) AS pickup_fhvhv_trip_count,
                SUM(source_od_row_count) AS pickup_source_od_row_count,
                SUM(yellow_source_od_row_count) AS pickup_yellow_od_row_count,
                SUM(green_source_od_row_count) AS pickup_green_od_row_count,
                SUM(fhvhv_source_od_row_count) AS pickup_fhvhv_od_row_count,
                MIN(date) AS pickup_min_date,
                MAX(date) AS pickup_max_date,
                SUM(CASE WHEN daypart IS NULL THEN 1 ELSE 0 END)
                    AS pickup_missing_daypart_rows,
                SUM(CASE WHEN daypart_order IS NULL THEN 1 ELSE 0 END)
                    AS pickup_missing_daypart_order_rows,
                SUM(CASE WHEN temporal_bucket_order IS NULL THEN 1 ELSE 0 END)
                    AS pickup_missing_temporal_bucket_order_rows,
                SUM(CASE WHEN observation_sequence_id IS NULL THEN 1 ELSE 0 END)
                    AS pickup_missing_sequence_rows,
                SUM(CASE WHEN spatial_assignment_method IS NULL THEN 1 ELSE 0 END)
                    AS pickup_missing_spatial_method_rows,
                SUM(CASE WHEN spatial_assignment_source IS NULL THEN 1 ELSE 0 END)
                    AS pickup_missing_spatial_source_rows,
                SUM(CASE WHEN spatial_assignment_confidence IS NULL THEN 1 ELSE 0 END)
                    AS pickup_missing_spatial_confidence_rows,
                SUM(CASE WHEN canonical_geography IS NULL THEN 1 ELSE 0 END)
                    AS pickup_missing_canonical_geography_rows
            FROM read_parquet([{final_pickup_file_sql_list}])
            GROUP BY modality
        )
        SELECT
            COALESCE(source_summary.modality, pickup_summary.modality) AS modality,
            source_summary.source_trip_count,
            pickup_summary.pickup_trip_count,
            pickup_summary.pickup_trip_count - source_summary.source_trip_count
                AS pickup_trip_difference_vs_source,
            source_summary.source_yellow_trip_count,
            pickup_summary.pickup_yellow_trip_count,
            pickup_summary.pickup_yellow_trip_count - source_summary.source_yellow_trip_count
                AS yellow_trip_difference_vs_source,
            source_summary.source_green_trip_count,
            pickup_summary.pickup_green_trip_count,
            pickup_summary.pickup_green_trip_count - source_summary.source_green_trip_count
                AS green_trip_difference_vs_source,
            source_summary.source_fhvhv_trip_count,
            pickup_summary.pickup_fhvhv_trip_count,
            pickup_summary.pickup_fhvhv_trip_count - source_summary.source_fhvhv_trip_count
                AS fhvhv_trip_difference_vs_source,
            source_summary.source_od_row_count,
            pickup_summary.pickup_source_od_row_count,
            pickup_summary.pickup_source_od_row_count - source_summary.source_od_row_count
                AS source_od_row_difference_vs_source,
            source_summary.source_yellow_od_row_count,
            pickup_summary.pickup_yellow_od_row_count,
            source_summary.source_green_od_row_count,
            pickup_summary.pickup_green_od_row_count,
            source_summary.source_fhvhv_od_row_count,
            pickup_summary.pickup_fhvhv_od_row_count,
            source_summary.source_min_date,
            source_summary.source_max_date,
            pickup_summary.pickup_min_date,
            pickup_summary.pickup_max_date,
            pickup_summary.pickup_missing_daypart_rows,
            pickup_summary.pickup_missing_daypart_order_rows,
            pickup_summary.pickup_missing_temporal_bucket_order_rows,
            pickup_summary.pickup_missing_sequence_rows,
            pickup_summary.pickup_missing_spatial_method_rows,
            pickup_summary.pickup_missing_spatial_source_rows,
            pickup_summary.pickup_missing_spatial_confidence_rows,
            pickup_summary.pickup_missing_canonical_geography_rows,
            '{FINAL_TLC_CONTRACT_VALIDATION_STRATEGY}' AS validation_strategy,
            '{current_contract_build_strategy}' AS contract_build_strategy
        FROM source_summary
        FULL OUTER JOIN pickup_summary
            ON source_summary.modality = pickup_summary.modality
        ORDER BY modality
    """

    tlc_final_contract_validation_df = duckdb.sql(validation_query).df()

    numeric_cols = tlc_final_contract_validation_df.select_dtypes(include=["float"]).columns
    tlc_final_contract_validation_df[numeric_cols] = (
        tlc_final_contract_validation_df[numeric_cols].round(6)
    )

    tlc_final_contract_validation_df.to_csv(
        TLC_FINAL_CONTRACT_VALIDATION_PATH,
        index=False
    )

    print("Wrote final TLC contract validation:", TLC_FINAL_CONTRACT_VALIDATION_PATH)

else:

    tlc_final_contract_validation_df = pd.read_csv(
        TLC_FINAL_CONTRACT_VALIDATION_PATH
    )

    print("Loaded current final TLC contract validation:", TLC_FINAL_CONTRACT_VALIDATION_PATH)

expected_final_modalities = {"taxi", "fhvhv"}
observed_final_modalities = set(tlc_final_contract_validation_df["modality"].tolist())

assert observed_final_modalities == expected_final_modalities, (
    "Final TLC modalities do not match the expected Taxi/FHVHV contract. "
    f"Observed: {sorted(observed_final_modalities)}"
)

contract_assertion_columns = [
    "pickup_trip_difference_vs_source",
    "yellow_trip_difference_vs_source",
    "green_trip_difference_vs_source",
    "fhvhv_trip_difference_vs_source",
    "source_od_row_difference_vs_source",
    "pickup_missing_daypart_rows",
    "pickup_missing_daypart_order_rows",
    "pickup_missing_temporal_bucket_order_rows",
    "pickup_missing_sequence_rows",
    "pickup_missing_spatial_method_rows",
    "pickup_missing_spatial_source_rows",
    "pickup_missing_spatial_confidence_rows",
    "pickup_missing_canonical_geography_rows",
]

failed_check_records = []

for _, row in tlc_final_contract_validation_df.iterrows():
    modality = row["modality"]

    for column in contract_assertion_columns:
        value = row[column]

        if pd.isna(value) or value != 0:
            failed_check_records.append({
                "modality": modality,
                "failed_check": column,
                "value": value,
            })

tlc_final_contract_failed_checks_df = pd.DataFrame(failed_check_records)

if not tlc_final_contract_failed_checks_df.empty:
    print("Final TLC contract validation failed checks:")
    display(tlc_final_contract_failed_checks_df)

assert tlc_final_contract_failed_checks_df.empty, (
    "Final TLC pickup-temporal contract validation failed. "
    "See tlc_final_contract_failed_checks_df for failing checks: "
    f"{tlc_final_contract_failed_checks_df['failed_check'].drop_duplicates().tolist()}"
)

pickup_min_date = pd.to_datetime(
    tlc_final_contract_validation_df["pickup_min_date"]
).min()
pickup_max_date = pd.to_datetime(
    tlc_final_contract_validation_df["pickup_max_date"]
).max()

if pickup_min_date < pd.Timestamp("2023-01-01"):
    print("Final TLC contract starts before study period:", pickup_min_date)

assert pickup_min_date >= pd.Timestamp("2023-01-01"), (
    "Final TLC contract starts before the study period. "
    f"Observed min date: {pickup_min_date}"
)

if pickup_max_date > pd.Timestamp("2026-03-31"):
    print("Final TLC contract extends beyond study period:", pickup_max_date)

assert pickup_max_date <= pd.Timestamp("2026-03-31"), (
    "Final TLC contract extends beyond the study period. "
    f"Observed max date: {pickup_max_date}"
)

print("Final TLC Taxi/FHVHV pickup-temporal contract validation passed.")

display(tlc_final_contract_validation_df)

Loaded current final TLC contract validation: pipeline_data/1.2.3.intermediate_tables/tlc_final_contract_validation.csv
Final TLC Taxi/FHVHV pickup-temporal contract validation passed.


,modality,source_trip_count,pickup_trip_count,pickup_trip_difference_vs_source,source_yellow_trip_count,pickup_yellow_trip_count,yellow_trip_difference_vs_source,source_green_trip_count,pickup_green_trip_count,green_trip_difference_vs_source,...,pickup_missing_daypart_rows,pickup_missing_daypart_order_rows,pickup_missing_temporal_bucket_order_rows,pickup_missing_sequence_rows,pickup_missing_spatial_method_rows,pickup_missing_spatial_source_rows,pickup_missing_spatial_confidence_rows,pickup_missing_canonical_geography_rows,validation_strategy,contract_build_strategy
0,fhvhv,778288736.0,778288736.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,service_month_filtered_monthly_pickup_temporal...,service_month_filtered_monthly_pickup_temporal_v3
1,taxi,139075726.0,139075726.0,0.0,136984940.0,136984940.0,0.0,2090786.0,2090786.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,service_month_filtered_monthly_pickup_temporal...,service_month_filtered_monthly_pickup_temporal_v3


Findings\. Validation passed for the final TLC pickup\-temporal contract\. The rebuilt monthly TLC mobility outputs are now service\-month clean, and the final Taxi/FHVHV contract preserves trip totals and source OD row counts exactly\. Taxi now combines Yellow and Green upstream with no trip loss, FHVHV remains separate, all outputs stay within the January 1, 2023 through March 31, 2026 study period, and the required temporal ordering and spatial provenance fields are fully populated\.

## 1\.2\.3\.5 Metric Validation

Before treating the TLC mobility tables as analysis\-ready, we need to validate the derived mobility metrics themselves\. Temporal fields can be present and zone IDs can map correctly, but impossible values can still slip through from the raw trip records and distort downstream averages, speeds, and temporal features\. This section checks for invalid duration and distance values after aggregation so we can catch upstream data\-quality issues before they flow into harmonization, merging, or feature engineering\.

In [21]:
# ---------------------------------------------------------------------
# Validate TLC mobility metric quality
# ---------------------------------------------------------------------

TLC_MOBILITY_METRIC_QUALITY_MANIFEST_PATH = (
    TLC_INTERMEDIATE_OUTPUT_DIR / "tlc_mobility_metric_quality_manifest.csv"
)

if (
    RUN_TLC_MOBILITY_METRIC_QUALITY_VALIDATION
    or not TLC_MOBILITY_METRIC_QUALITY_MANIFEST_PATH.exists()
):

    query = f"""
        SELECT
            dataset_type,

            COUNT(*) AS row_count,

            SUM(CASE WHEN avg_trip_duration_seconds < 0 THEN 1 ELSE 0 END)
                AS negative_duration_rows,

            SUM(CASE WHEN avg_trip_duration_seconds = 0 THEN 1 ELSE 0 END)
                AS zero_duration_rows,

            SUM(CASE WHEN avg_trip_duration_seconds > {MAX_VALID_TLC_TRIP_DURATION_SECONDS} THEN 1 ELSE 0 END)
                AS excessive_duration_rows,

            SUM(CASE WHEN avg_trip_distance < 0 THEN 1 ELSE 0 END)
                AS negative_distance_rows,

            SUM(CASE WHEN avg_trip_distance > {MAX_VALID_TLC_TRIP_DISTANCE} THEN 1 ELSE 0 END)
                AS excessive_distance_rows,

            SUM(
                CASE
                    WHEN avg_trip_duration_seconds > 0
                     AND (avg_trip_distance / (avg_trip_duration_seconds / 3600.0)) > {MAX_VALID_TLC_TRIP_SPEED_MPH}
                    THEN 1 ELSE 0
                END
            ) AS excessive_speed_rows,

            MIN(avg_trip_duration_seconds) AS min_avg_trip_duration_seconds,
            MAX(avg_trip_duration_seconds) AS max_avg_trip_duration_seconds,

            MIN(avg_trip_distance) AS min_avg_trip_distance,
            MAX(avg_trip_distance) AS max_avg_trip_distance,

            MIN(
                CASE
                    WHEN avg_trip_duration_seconds > 0
                    THEN avg_trip_distance / (avg_trip_duration_seconds / 3600.0)
                    ELSE NULL
                END
            ) AS min_implied_avg_trip_speed,

            MAX(
                CASE
                    WHEN avg_trip_duration_seconds > 0
                    THEN avg_trip_distance / (avg_trip_duration_seconds / 3600.0)
                    ELSE NULL
                END
            ) AS max_implied_avg_trip_speed

        FROM read_parquet(
            '{TLC_MOBILITY_OUTPUT_DIR.as_posix()}/*_mobility.parquet'
        )

        GROUP BY dataset_type
        ORDER BY dataset_type
    """

    tlc_mobility_metric_quality_df = duckdb.sql(query).df()

    tlc_mobility_metric_quality_df.to_csv(
        TLC_MOBILITY_METRIC_QUALITY_MANIFEST_PATH,
        index=False
    )

    print(
        "Wrote TLC mobility metric quality manifest:",
        TLC_MOBILITY_METRIC_QUALITY_MANIFEST_PATH
    )

else:

    tlc_mobility_metric_quality_df = pd.read_csv(
        TLC_MOBILITY_METRIC_QUALITY_MANIFEST_PATH
    )

    print(
        "Loaded existing TLC mobility metric quality manifest:",
        TLC_MOBILITY_METRIC_QUALITY_MANIFEST_PATH
    )

display(tlc_mobility_metric_quality_df)

Loaded existing TLC mobility metric quality manifest: pipeline_data/1.2.3.intermediate_tables/tlc_mobility_metric_quality_manifest.csv


,dataset_type,row_count,negative_duration_rows,zero_duration_rows,excessive_duration_rows,negative_distance_rows,excessive_distance_rows,excessive_speed_rows,min_avg_trip_duration_seconds,max_avg_trip_duration_seconds,min_avg_trip_distance,max_avg_trip_distance,min_implied_avg_trip_speed,max_implied_avg_trip_speed
0,fhvhv,272681609,0.0,0.0,0.0,0.0,0.0,0.0,60.0,72373.0,0.0,100.00,0.0,98.919196
1,green,1639425,0.0,0.0,0.0,0.0,0.0,0.0,60.0,86397.0,0.0,99.28,0.0,99.911392
2,yellow,53542693,0.0,0.0,0.0,0.0,0.0,0.0,60.0,86399.0,0.0,99.91,0.0,100.000000


In [22]:
# ---------------------------------------------------------------------
# Assert TLC mobility metric quality
# ---------------------------------------------------------------------

invalid_metric_rows = (
    tlc_mobility_metric_quality_df[
        [
            "negative_duration_rows",
            "zero_duration_rows",
            "excessive_duration_rows",
            "negative_distance_rows",
            "excessive_distance_rows",
            "excessive_speed_rows",
        ]
    ]
    .sum()
    .sum()
)

assert invalid_metric_rows == 0, (
    "TLC mobility metric quality validation failed. "
    "Review tlc_mobility_metric_quality_df for invalid duration, distance, or speed values."
)

print("TLC mobility metric quality validation passed.")

TLC mobility metric quality validation passed.


Findings\. The metric\-quality check passed once invalid trips were filtered before aggregation\. That means the retained TLC mobility tables no longer carry impossible durations, impossible distances, or impossible implied speeds into downstream averages\. The bad values were coming from a small set of raw trip records, and that problem is now handled upstream instead of leaking into later notebooks\.

### Summary

The aggregated TLC mobility tables retain essentially complete spatial coverage of the Taxi Zone system\. No missing pickup or dropoff zones were introduced during aggregation, and the resulting mobility tables continue to represent activity across nearly the entire Taxi Zone universe\. A small number of pickup zone identifiers \(57, 105, 264, and 265\) did not resolve against the current Spatial Reference Layer, but together account for less than 0\.1% of retained trips\. Overall, the results indicate that the aggregation process preserved the spatial structure of the TLC data and produced mobility tables suitable for downstream Taxi Zone\-level analysis\.

> 🔍 Cross\-modal Coverage Insight\. One of the benefits of combining multiple mobility systems is that each dataset observes a different portion of the city\. Roadway traffic sensors cover only 138 Taxi Zones, reflecting the limited footprint of the traffic monitoring network\. Bus activity spans 252 Taxi Zones, providing substantially broader geographic coverage, while TLC activity reaches essentially the entire Taxi Zone system with 263 pickup zones and 264 dropoff zones represented\. Subway activity contributes coverage across 156 Taxi Zones and, while more geographically concentrated than Bus or TLC, provides deep visibility into some of the city's most important transit corridors, employment centers, and commercial districts\. Although no single mobility source provides complete visibility into NYC transportation activity, the combined multimodal dataset offers far more comprehensive geographic coverage than any individual modality alone\. This complementary coverage is one of the primary motivations for constructing an integrated mobility panel and may ultimately improve the robustness of downstream anomaly detection and congestion\-pricing analyses\.

### Aggregation Integrity

The most important check for this notebook is whether aggregation preserved the retained TLC trip volume\. Because invalid\-trip filtering is enabled, the correct comparison is not all raw trips versus aggregated trips\. The correct comparison is valid raw trips after filtering versus summed trip\_count values in the mobility tables\. This confirms that aggregation did not accidentally drop or duplicate trips within the retained valid trip universe\.

In [23]:
# ---------------------------------------------------------------------
# Validate raw-to-monthly TLC mobility aggregation integrity
# ---------------------------------------------------------------------

REBUILD_TLC_AGGREGATION_INTEGRITY = False
TLC_AGGREGATION_INTEGRITY_STRATEGY = "service_month_filtered_valid_trip_integrity_v2"

TLC_AGGREGATION_INTEGRITY_PATH = (
    TLC_INTERMEDIATE_OUTPUT_DIR / "tlc_aggregation_integrity.parquet"
)

aggregation_integrity_cache_is_current = False

if TLC_AGGREGATION_INTEGRITY_PATH.exists():
    cached_aggregation_integrity_df = pd.read_parquet(TLC_AGGREGATION_INTEGRITY_PATH)

    aggregation_integrity_cache_is_current = (
        "validation_strategy" in cached_aggregation_integrity_df.columns
        and cached_aggregation_integrity_df["validation_strategy"]
            .eq(TLC_AGGREGATION_INTEGRITY_STRATEGY)
            .all()
    )

if (
    TLC_AGGREGATION_INTEGRITY_PATH.exists()
    and aggregation_integrity_cache_is_current
    and not REBUILD_TLC_AGGREGATION_INTEGRITY
):
    aggregation_integrity_df = cached_aggregation_integrity_df
    print(
        "Loaded cached TLC aggregation integrity summary:",
        TLC_AGGREGATION_INTEGRITY_PATH
    )

else:
    if TLC_AGGREGATION_INTEGRITY_PATH.exists() and not aggregation_integrity_cache_is_current:
        print(
            "Cached TLC aggregation integrity summary is stale; rebuilding against service-month filtered raw trips."
        )

    aggregation_integrity_records = []

    for dataset_type, config in retained_tlc_datasets.items():

        source_files = sorted(
            TLC_DATA_DIR.glob(f"{dataset_type}_tripdata_*.parquet")
        )

        output_files = sorted(
            TLC_MOBILITY_OUTPUT_DIR.glob(
                f"{dataset_type}_tripdata_*_mobility.parquet"
            )
        )

        distance_col = config["distance_col"]
        duration_seconds_expr = config["duration_seconds_expr"]
        pickup_datetime_col = config["pickup_datetime_col"]

        trip_quality_filter = f"""
            {duration_seconds_expr} BETWEEN {MIN_VALID_TLC_TRIP_DURATION_SECONDS}
                AND {MAX_VALID_TLC_TRIP_DURATION_SECONDS}
            AND "{distance_col}" BETWEEN {MIN_VALID_TLC_TRIP_DISTANCE}
                AND {MAX_VALID_TLC_TRIP_DISTANCE}
            AND (
                "{distance_col}" / ({duration_seconds_expr} / 3600.0)
            ) <= {MAX_VALID_TLC_TRIP_SPEED_MPH}
        """

        raw_trip_count = 0
        service_month_raw_trip_count = 0
        retained_valid_trip_count = 0
        out_of_service_month_raw_trip_count = 0

        for source_path in source_files:

            service_month = (
                source_path.name
                .split("_tripdata_")[1]
                .replace(".parquet", "")
            )

            service_month_start = pd.Timestamp(f"{service_month}-01")
            next_service_month_start = service_month_start + pd.offsets.MonthBegin(1)

            service_month_filter = f"""
                CAST("{pickup_datetime_col}" AS DATE) >= DATE '{service_month_start.date()}'
                AND CAST("{pickup_datetime_col}" AS DATE) < DATE '{next_service_month_start.date()}'
            """

            raw_trip_count += duckdb.sql(f"""
                SELECT COUNT(*) AS trips
                FROM read_parquet('{source_path.as_posix()}')
            """).fetchone()[0]

            service_month_raw_trip_count += duckdb.sql(f"""
                SELECT COUNT(*) AS trips
                FROM read_parquet('{source_path.as_posix()}')
                WHERE {service_month_filter}
            """).fetchone()[0]

            out_of_service_month_raw_trip_count += duckdb.sql(f"""
                SELECT COUNT(*) AS trips
                FROM read_parquet('{source_path.as_posix()}')
                WHERE NOT ({service_month_filter})
            """).fetchone()[0]

            if FILTER_INVALID_TLC_TRIPS:
                retained_valid_trip_count += duckdb.sql(f"""
                    SELECT COUNT(*) AS trips
                    FROM read_parquet('{source_path.as_posix()}')
                    WHERE ({service_month_filter})
                      AND ({trip_quality_filter})
                """).fetchone()[0]
            else:
                retained_valid_trip_count += duckdb.sql(f"""
                    SELECT COUNT(*) AS trips
                    FROM read_parquet('{source_path.as_posix()}')
                    WHERE {service_month_filter}
                """).fetchone()[0]

        mobility_table_trip_count = 0
        mobility_state_count = 0

        for output_path in output_files:
            result = duckdb.sql(f"""
                SELECT
                    COUNT(*) AS mobility_states,
                    SUM(trip_count) AS trips
                FROM read_parquet('{output_path.as_posix()}')
            """).df()

            mobility_state_count += result["mobility_states"].iloc[0]
            mobility_table_trip_count += result["trips"].iloc[0]

        filtered_trip_count = service_month_raw_trip_count - retained_valid_trip_count

        aggregation_integrity_records.append({
            "dataset_type": dataset_type,
            "validation_strategy": TLC_AGGREGATION_INTEGRITY_STRATEGY,
            "source_files": len(source_files),
            "output_files": len(output_files),
            "raw_trip_count": raw_trip_count,
            "service_month_raw_trip_count": service_month_raw_trip_count,
            "out_of_service_month_raw_trip_count": out_of_service_month_raw_trip_count,
            "retained_valid_trip_count": retained_valid_trip_count,
            "filtered_trip_count": filtered_trip_count,
            "filtered_trip_pct": (
                filtered_trip_count / service_month_raw_trip_count
                if service_month_raw_trip_count > 0 else None
            ),
            "mobility_table_trip_count": mobility_table_trip_count,
            "trip_count_difference_vs_retained_valid": (
                mobility_table_trip_count - retained_valid_trip_count
            ),
            "mobility_state_count": mobility_state_count,
            "compression_ratio_rows": (
                retained_valid_trip_count / mobility_state_count
                if mobility_state_count > 0 else None
            ),
        })

    aggregation_integrity_df = pd.DataFrame(aggregation_integrity_records)

    float_cols = aggregation_integrity_df.select_dtypes(include=["float"]).columns
    aggregation_integrity_df[float_cols] = (
        aggregation_integrity_df[float_cols]
        .round(6)
    )

    aggregation_integrity_df.to_parquet(
        TLC_AGGREGATION_INTEGRITY_PATH,
        index=False,
    )

    print(
        "Wrote TLC aggregation integrity summary:",
        TLC_AGGREGATION_INTEGRITY_PATH
    )

failed_aggregation_integrity_df = aggregation_integrity_df[
    ~aggregation_integrity_df["trip_count_difference_vs_retained_valid"].eq(0)
]

if not failed_aggregation_integrity_df.empty:
    print("TLC aggregation integrity failed checks:")
    display(failed_aggregation_integrity_df)

assert failed_aggregation_integrity_df.empty, (
    "TLC aggregation integrity failed: mobility table trip counts do not match "
    "service-month filtered retained valid raw trip counts."
)

aggregation_integrity_df

Loaded cached TLC aggregation integrity summary: pipeline_data/1.2.3.intermediate_tables/tlc_aggregation_integrity.parquet


,dataset_type,validation_strategy,source_files,output_files,raw_trip_count,service_month_raw_trip_count,out_of_service_month_raw_trip_count,retained_valid_trip_count,filtered_trip_count,filtered_trip_pct,mobility_table_trip_count,trip_count_difference_vs_retained_valid,mobility_state_count,compression_ratio_rows
0,yellow,service_month_filtered_valid_trip_integrity_v2,40,40,143080418,143079000,1418,143079000,0,0.0,143079000.0,0.0,53987903,2.650205
1,green,service_month_filtered_valid_trip_integrity_v2,39,39,2160506,2159911,595,2159911,0,0.0,2159911.0,0.0,1690028,1.278033
2,fhvhv,service_month_filtered_valid_trip_integrity_v2,39,39,778424569,778424569,0,778424569,0,0.0,778424569.0,0.0,272691530,2.854598


Findings\. The retained\-valid integrity check passed\. The mobility\-table trip totals match the raw TLC trips that survived the duration, distance, and implied\-speed filters, with no remaining aggregation gap against the valid\-trip universe\. This confirms the tables are not losing valid trips during aggregation and are no longer being judged against trips that were intentionally filtered out\.

The results also show how much compression the aggregation creates after filtering\. The raw TLC files contain hundreds of millions of valid trip records, while the mobility tables collapse those trips into a much smaller set of origin\-destination mobility states\. The compression ratio should be interpreted within the retained valid trip universe, not the full unfiltered source universe\.

### Top Pickup\-Zone Check

In [24]:

# ---------------------------------------------------------------------
# Top pickup zones by modality
# ---------------------------------------------------------------------

for dataset_type in retained_tlc_datasets:

    print("=" * 80)
    print(dataset_type.upper())
    print("=" * 80)

    top_pickup_zones = duckdb.sql(f"""
        SELECT
            pickup_zone_id,
            SUM(trip_count) AS trips
        FROM read_parquet(
            '{TLC_MOBILITY_OUTPUT_DIR.as_posix()}/{dataset_type}_tripdata_*_mobility.parquet'
        )
        GROUP BY 1
        ORDER BY trips DESC
        LIMIT 20
    """).df()

    top_pickup_zones = top_pickup_zones.merge(
        taxi_zones_lookup,
        left_on="pickup_zone_id",
        right_on="taxi_zone_id",
        how="left"
    )

    display(
        top_pickup_zones[
            [
                "pickup_zone_id",
                "taxi_zone_name",
                "taxi_zone_borough",
                "trips",
            ]
        ]
    )

YELLOW


,pickup_zone_id,taxi_zone_name,taxi_zone_borough,trips
0,132,JFK Airport,Queens,6623605.0
1,237,Upper East Side South,Manhattan,6493297.0
2,161,Midtown Center,Manhattan,6379736.0
3,236,Upper East Side North,Manhattan,5810498.0
4,162,Midtown East,Manhattan,4727398.0
5,186,Penn Station/Madison Sq West,Manhattan,4657309.0
6,230,Times Sq/Theatre District,Manhattan,4602638.0
7,142,Lincoln Square East,Manhattan,4385537.0
8,138,LaGuardia Airport,Queens,4247914.0
9,170,Murray Hill,Manhattan,3983223.0


GREEN


,pickup_zone_id,taxi_zone_name,taxi_zone_borough,trips
0,74,East Harlem North,Manhattan,485451.0
1,75,East Harlem South,Manhattan,291823.0
2,166,Morningside Heights,Manhattan,106251.0
3,95,Forest Hills,Queens,105988.0
4,41,Central Harlem,Manhattan,100116.0
5,43,Central Park,Manhattan,97666.0
6,82,Elmhurst,Queens,89743.0
7,97,Fort Greene,Brooklyn,68437.0
8,65,Downtown Brooklyn/MetroTech,Brooklyn,59260.0
9,130,Jamaica,Queens,55322.0


FHVHV


,pickup_zone_id,taxi_zone_name,taxi_zone_borough,trips
0,138,LaGuardia Airport,Queens,15152270.0
1,132,JFK Airport,Queens,13728458.0
2,61,Crown Heights North,Brooklyn,10161342.0
3,79,East Village,Manhattan,10093722.0
4,230,Times Sq/Theatre District,Manhattan,9432117.0
5,161,Midtown Center,Manhattan,9221325.0
6,231,TriBeCa/Civic Center,Manhattan,8776720.0
7,68,East Chelsea,Manhattan,8529894.0
8,76,East New York,Brooklyn,8519698.0
9,37,Bushwick South,Brooklyn,8501774.0


Findings\. The top pickup zones look like what we would expect from the three TLC services, which is a reassuring sanity check\. Yellow Taxi is concentrated in major Manhattan activity centers and airports, Green Taxi leans more toward Upper Manhattan plus parts of Queens and Brooklyn, and FHVHV is broader across dense residential areas, commercial centers, and airports\. Nothing here suggests a bad geographic join or a strange aggregation artifact\.

## Close

This notebook converts the retained TLC trip datasets into compact mobility tables and then resolves the final downstream TLC contract\. The monthly origin\-destination tables remain available as the detail and audit layer, but the main handoff is now the compact two\-modality pickup\-temporal structure: Taxi, which merges Yellow and Green upstream during final aggregation, and FHVHV as its own separate modality\.

Validation also confirms the parts that matter most for downstream use\. The final TLC handoff preserves retained valid trip volume, keeps pickup/dropoff detail long enough to compute destination diversity the right way, carries forward the ordering fields added in \`1\.2\.1\`, and keeps the native TLC Taxi Zone provenance from \`1\.2\.2\`\. Distance, duration, and speed are all rebuilt from valid aggregate numerators and denominators rather than from averaging pre\-aggregated means\.

From here on, downstream harmonization and panel\-building work should read the final pickup\-temporal TLC contract directly\. The monthly OD outputs still matter, but mainly as the audit/detail layer for destination\-diversity checks and any future flow\-oriented app work\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5c43663c-345d-4c21-9332-dd4f928af3ba' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>